# Notebook de Experimentos — Feature Engineering & Mejoras de Modelo

**Flujo general:**
1. [CELDA 1] Imports y utilidades — ejecutar siempre primero
2. [CELDA 2] Carga de artefactos base (X_train, y_train, splits...)
3. [CELDA 3] Carga del dataset limpio (df_original, df_train, df_test)
4. [CELDA 4] Carga del dataset completo 88 features (df_88_train, df_88_test)
5. [CELDA 5] Carga de artefactos exp4 y exp5b — punto de partida de mejoras
6. [CELDA 6–10] Experimentos de features (exp1→exp5b)
7. [CELDA 11] Validación test holdout exp5b
8. [CELDA 12] EXP-TH — Threshold Tuning
9. [CELDA 13] EXP-ST — Stacking LGB + XGB + CAT
10. [CELDA 14] EXP-CA — Cascada urgente / no urgente
11. [CELDA 15] Resumen comparativo global

**Regla de oro:** los artefactos se exportan en el mismo directorio del notebook (`NOTEBOOK_DIR`).  
Si un experimento no supera el umbral de ganancia, no se exporta nada.

## CELDA 1 — Imports y utilidades
*Ejecutar siempre primero. Define todas las dependencias y helpers.*

In [27]:
# =========================================================
# CELDA 1 — Imports y utilidades
# =========================================================
from pathlib import Path
from scipy.stats import kruskal
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report, precision_recall_fscore_support
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from lightgbm import LGBMClassifier
import xgboost as xgb
import catboost as cb
import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings("ignore")

# Directorio del notebook — artefactos se guardan aquí
NOTEBOOK_DIR = Path.cwd()

# ── Helpers ──────────────────────────────────────────────
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

def aplicar_thresholds(proba, w):
    """Aplica pesos por clase a probabilidades y devuelve argmax (0-indexed)."""
    return np.argmax(proba * w, axis=1)

def objetivo_th(w, proba, y_true):
    return -f1_score(y_true, aplicar_thresholds(proba, w),
                     average="macro", zero_division=0)

print("✅ Imports OK")
print(f"   Directorio de artefactos: {NOTEBOOK_DIR}")


✅ Imports OK
   Directorio de artefactos: c:\Users\CARLOS\triaje-ia-tfg\notebooks\3_modeling


## CELDA 2 — Carga de artefactos base
*X_train, y_train, groups, weights, params del modelo tuneado. Requiere CELDA 1.*

In [28]:
# =========================================================
# CELDA 2 — Artefactos base (read-only)
# =========================================================
from triaje_ia.config import DATA_PROCESSED, MODELS_DIR

X_train = pd.read_parquet(DATA_PROCESSED / "X_train.parquet")
X_test  = pd.read_parquet(DATA_PROCESSED / "X_test.parquet")
y_train = pd.read_parquet(DATA_PROCESSED / "y_train.parquet").squeeze()
y_test  = pd.read_parquet(DATA_PROCESSED / "y_test.parquet").squeeze()

groups_train         = np.load(DATA_PROCESSED / "groups_train.npy")
sample_weights_train = np.load(DATA_PROCESSED / "sample_weights_train.npy")

with open(DATA_PROCESSED / "feature_config_selected.json", encoding="utf-8") as f:
    feature_config = json.load(f)

FEATURES_SELECCIONADAS = feature_config["features_seleccionadas"]   # 46 features
TARGET                 = feature_config["target"]

X_tr_sel = X_train[FEATURES_SELECCIONADAS].copy()
X_te_sel = X_test[FEATURES_SELECCIONADAS].copy()

# Parámetros del modelo tuneado
with open(MODELS_DIR / "lgbm_best_params.json", encoding="utf-8") as f:
    best_params = json.load(f)

params_modelo = {k: v for k, v in best_params.items() if not k.startswith("_")}

CV          = StratifiedGroupKFold(n_splits=5, shuffle=False)
f1_baseline = 0.4871   # baseline tuned 46 features

print(f"X_train: {X_train.shape} | X_tr_sel: {X_tr_sel.shape}")
print(f"y_train: {y_train.shape} | clases: {sorted(y_train.unique())}")
print(f"Params modelo: {list(params_modelo.keys())}")


X_train: (334480, 46) | X_tr_sel: (334480, 46)
y_train: (334480,) | clases: [np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(5)]
Params modelo: ['num_leaves', 'max_depth', 'min_data_in_leaf', 'reg_alpha', 'reg_lambda', 'min_gain_to_split', 'subsample', 'colsample_bytree', 'learning_rate', 'objective', 'num_class', 'metric', 'n_estimators', 'subsample_freq', 'random_state', 'n_jobs', 'verbose']


## CELDA 3 — Dataset limpio (df_original)
*Necesario para construir features v2 y experimentos de vitales. Requiere CELDA 2.*

In [3]:
# =========================================================
# CELDA 3 — Carga dataset limpio y split temporal
# =========================================================
from triaje_ia.data.cleaner import cargar_dataset_limpio

df_original = cargar_dataset_limpio(forzar=False)

fecha_corte = df_original["intime"].quantile(0.80)
mask_train  = df_original["intime"] <= fecha_corte
mask_test   = df_original["intime"] >  fecha_corte

df_train = df_original[mask_train].reset_index(drop=True)
df_test  = df_original[mask_test].reset_index(drop=True)

assert len(df_train) == len(X_train), f"Desalineación train: {len(df_train)} vs {len(X_train)}"
assert len(df_test)  == len(X_test),  f"Desalineación test:  {len(df_test)} vs {len(X_test)}"

print(f"df_original : {df_original.shape}")
print(f"df_train    : {df_train.shape} ✅")
print(f"df_test     : {df_test.shape}  ✅")


2026-05-16 00:03:13.040 | WARNING  | triaje_ia.data.cleaner:cargar_dataset_limpio:193 - Cargando desde caché: C:\Users\CARLOS\triaje-ia-tfg\data\interim\dataset_clean.parquet. Si cambiaste el pipeline, usa forzar=True.
2026-05-16 00:03:13.549 | SUCCESS  | triaje_ia.data.cleaner:cargar_dataset_limpio:198 - Cargado: 418,100 filas | 21 columnas


df_original : (418100, 21)
df_train    : (334480, 21) ✅
df_test     : (83620, 21)  ✅


## CELDA 4 — Dataset completo 88 features
*Necesario para kitchen sink y ablación. Requiere CELDA 2.*

In [4]:
# =========================================================
# CELDA 4 — Dataset completo 88 features
# =========================================================
dataset_completo = pd.read_parquet(DATA_PROCESSED / "dataset_features.parquet")

fecha_corte_88 = df_original["intime"].quantile(0.80)
mask_tr_88     = df_original["intime"] <= fecha_corte_88
mask_te_88     = df_original["intime"] >  fecha_corte_88

df_88_train = dataset_completo[mask_tr_88.values].reset_index(drop=True)
df_88_test  = dataset_completo[mask_te_88.values].reset_index(drop=True)

assert len(df_88_train) == len(X_train), f"Desalineación train: {len(df_88_train)} vs {len(X_train)}"
assert len(df_88_test)  == len(X_test),  f"Desalineación test:  {len(df_88_test)} vs {len(X_test)}"

GRUPOS_DESCARTADAS = {
    "vitales_flags": [
        "fiebre", "hipotension", "hipertension_severa",
        "qsofa_positivo", "shock_index_severo", "anciano", "anciano_mayor",
    ],
    "visitas": [
        "n_visitas_previas", "visitas_ultimo_mes", "visitas_ultimo_año",
        "dias_desde_ultima_visita", "primera_visita", "frecuentador",
    ],
    "cc_extra": [
        "cc_cefalea", "cc_intoxicacion", "cc_infeccioso", "cc_hemorragia_activa",
    ],
    "hx_extra": [
        "hx_respiratorio", "hx_neuro", "hx_psiquiatrico", "hx_abuso_sustancias",
        "hx_digestivo", "hx_metabolico_renal", "hx_infeccioso", "hx_trauma_muscular",
    ],
    "med_extra": [
        "med_anticoagulante", "med_antidiabetico_oral", "med_corticoide_sistemico",
        "med_benzodiacepina", "med_diuretico_tiazida", "med_respiratorio_inhalado",
        "med_inmunosupresor", "med_digoxina", "med_antiaritmico",
        "riesgo_depresion_resp", "polifarmacia", "hiperpolifarmacia",
    ],
    "llegada_extra": ["llegada_autonoma"],
}

print(f"Dataset completo: {dataset_completo.shape}")
print(f"df_88_train: {df_88_train.shape} ✅")
print(f"df_88_test : {df_88_test.shape}  ✅")


Dataset completo: (418100, 89)
df_88_train: (334480, 89) ✅
df_88_test : (83620, 89)  ✅


## CELDA 5 — Cargar artefactos exp4 y exp5b
*Punto de partida para todos los experimentos de mejora. Requiere CELDA 2.*

In [29]:
# =========================================================
# CELDA 5 — Cargar exp4 (49f) y exp5b (67f) desde disco
# =========================================================
# Si los parquets no existen aún, ejecutar primero las
# celdas 6–10 para generarlos.
# =========================================================

X_tr_exp4 = pd.read_parquet(DATA_PROCESSED / "X_train_exp4.parquet")
X_te_exp4 = pd.read_parquet(DATA_PROCESSED / "X_test_exp4.parquet")

with open(DATA_PROCESSED / "feature_config_exp4.json", encoding="utf-8") as f:
    cfg_exp4 = json.load(f)
f1_exp4 = cfg_exp4["macro_f1_exp4"]   # 0.4931

X_tr_exp5b = pd.read_parquet(DATA_PROCESSED / "X_train_exp5b.parquet")
X_te_exp5b = pd.read_parquet(DATA_PROCESSED / "X_test_exp5b.parquet")

with open(DATA_PROCESSED / "feature_config_exp5b.json", encoding="utf-8") as f:
    cfg_exp5b = json.load(f)
FEATURES_EXP5B = cfg_exp5b["todas_features"]
f1_exp5b       = cfg_exp5b["macro_f1_exp5b"]   # 0.5198

print(f"exp4  — X_train: {X_tr_exp4.shape}  |  CV F1: {f1_exp4}")
print(f"exp5b — X_train: {X_tr_exp5b.shape} |  CV F1: {f1_exp5b}")


exp4  — X_train: (334480, 49)  |  CV F1: 0.493085
exp5b — X_train: (334480, 67) |  CV F1: 0.51983


## CELDA 6 — EXP1: Features de interacción (v2)
*Construye 14 features de interacción clínica. Requiere CELDAS 2 y 3.*

In [6]:
# =========================================================
# EXP1 — Features de interacción clínica (v2)
# =========================================================

def construir_features_v2(df, X):
    feats = pd.DataFrame(index=df.index)
    pain          = X["pain"].fillna(0)
    age           = X["age"].fillna(0)
    n_vit         = X["n_vitales_anomalos"].fillna(0)
    n_med         = X["n_medicamentos"].fillna(0)
    zona_verde    = X["zona_verde"].fillna(0)
    sin_med       = X["sin_medicacion"].fillna(0)
    o2sat_bajo    = X["o2sat_bajo_92"].fillna(0)
    news2_alto    = X["news2_alto"].fillna(0)
    shock_alto    = X["shock_index_alto"].fillna(0)
    cc_dolor      = X["cc_dolor_toracico"].fillna(0)
    cc_disnea     = X["cc_disnea"].fillna(0)
    cc_neuro      = X["cc_neuro_ams"].fillna(0)
    cc_trauma     = X["cc_trauma"].fillna(0)
    med_opiaceo   = X["med_opiaceo"].fillna(0)
    alto_sangrado = X["alto_riesgo_sangrado"].fillna(0)
    llegada_autonoma = df["arrival_transport"].isin(
        ["WALK IN", "SELF", "AMBULATORY"]).astype("int8")

    feats["perfil_no_urgente"]       = ((zona_verde==1)&(sin_med==1)&(llegada_autonoma==1)&(pain<=3)&(n_vit==0)).astype("int8")
    feats["cronico_estable"]         = ((n_med>=5)&(zona_verde==1)&(llegada_autonoma==1)).astype("int8")
    feats["dolor_toracico_critico"]  = ((cc_dolor==1)&(shock_alto==1)).astype("int8")
    feats["dolor_toracico_estable"]  = ((cc_dolor==1)&(zona_verde==1)).astype("int8")
    feats["disnea_hipoxia"]          = ((cc_disnea==1)&(o2sat_bajo==1)).astype("int8")
    feats["disnea_compensada"]       = ((cc_disnea==1)&(o2sat_bajo==0)&(zona_verde==1)).astype("int8")
    feats["trauma_anciano"]          = ((cc_trauma==1)&(age>=65)).astype("int8")
    feats["neuro_anciano"]           = ((cc_neuro==1)&(age>=65)).astype("int8")
    feats["trauma_anticoagulado"]    = ((cc_trauma==1)&(alto_sangrado==1)).astype("int8")
    feats["pain_sin_deterioro"]      = ((pain>=7)&(news2_alto==0)).astype("int8")
    feats["deterioro_sin_dolor"]     = ((pain<=3)&(news2_alto==1)).astype("int8")
    feats["dolor_enmascarado_opiaceo"] = ((med_opiaceo==1)&(pain<=3)).astype("int8")
    cc_cols = [c for c in FEATURES_SELECCIONADAS if c.startswith("cc_")]
    feats["n_cc_activos"]            = X[cc_cols].fillna(0).sum(axis=1).astype("int8")
    feats["alarma_sin_expresion"]    = (((cc_dolor==1)|(cc_disnea==1)|(cc_neuro==1))&(zona_verde==1)&(n_vit==0)).astype("int8")
    return feats

feats_train_v2 = construir_features_v2(df_train, X_tr_sel)
feats_test_v2  = construir_features_v2(df_test,  X_te_sel)
FEATURES_NUEVAS_V2 = list(feats_train_v2.columns)

# Validación univariante
print("Validación univariante features v2:")
features_validas_v2 = []
for col in FEATURES_NUEVAS_V2:
    prev = feats_train_v2[col].mean()
    if feats_train_v2[col].std() > 0 and prev > 0.001:
        grupos = [feats_train_v2[col][y_train==k].values for k in sorted(y_train.unique())]
        H, _   = kruskal(*grupos)
        flag   = "✅" if H > 100 else ("⚠️ " if H > 20 else "🚨")
        print(f"  {flag} {col:<32} H={H:>8.1f}")
        if H > 20:
            features_validas_v2.append(col)

FEATURES_V2  = FEATURES_SELECCIONADAS + features_validas_v2
X_tr_v2      = pd.concat([X_tr_sel.reset_index(drop=True), feats_train_v2[features_validas_v2].reset_index(drop=True)], axis=1)
X_te_v2      = pd.concat([X_te_sel.reset_index(drop=True), feats_test_v2[features_validas_v2].reset_index(drop=True)],  axis=1)

scores_v2, n_trees_v2 = [], []
lgbm_v2 = LGBMClassifier(**params_modelo)
print(f"\nCV EXP1 ({len(FEATURES_V2)} features)...")
for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_v2, y_train, groups=groups_train)):
    lgbm_v2.fit(
        X_tr_v2.iloc[idx_tr], y_train.iloc[idx_tr]-1,
        sample_weight=sample_weights_train[idx_tr],
        eval_set=[(X_tr_v2.iloc[idx_val], y_train.iloc[idx_val]-1)],
        eval_sample_weight=[sample_weights_train[idx_val]],
        callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(period=-1)],
    )
    s = macro_f1(y_train.iloc[idx_val], lgbm_v2.predict(X_tr_v2.iloc[idx_val])+1)
    scores_v2.append(s); n_trees_v2.append(lgbm_v2.best_iteration_)
    print(f"  Fold {fold+1}: {s:.4f}  |  árboles: {lgbm_v2.best_iteration_}")

f1_v2    = np.mean(scores_v2)
ganancia = f1_v2 - f1_baseline
print(f"\n  EXP1 ({len(FEATURES_V2)}f): {f1_v2:.4f} ± {np.std(scores_v2):.4f}  Δbaseline={ganancia:+.4f}")


Validación univariante features v2:
  ✅ perfil_no_urgente                H=  4775.9
  ✅ cronico_estable                  H=  1733.5
  ✅ dolor_toracico_critico           H=   856.6
  ✅ dolor_toracico_estable           H=  2598.7
  ✅ disnea_hipoxia                   H=  6667.6
  ✅ disnea_compensada                H=  1335.2
  ✅ trauma_anciano                   H=   305.5
  ✅ neuro_anciano                    H=  9244.9
  ✅ trauma_anticoagulado             H=   179.6
  ✅ pain_sin_deterioro               H= 12105.4
  ✅ deterioro_sin_dolor              H=  5826.6
  ✅ dolor_enmascarado_opiaceo        H=  3413.0
  ✅ n_cc_activos                     H= 21813.2
  ✅ alarma_sin_expresion             H=  4697.4

CV EXP1 (60 features)...
  Fold 1: 0.4865  |  árboles: 389
  Fold 2: 0.4787  |  árboles: 348
  Fold 3: 0.4914  |  árboles: 358
  Fold 4: 0.4825  |  árboles: 360
  Fold 5: 0.4873  |  árboles: 371

  EXP1 (60f): 0.4853 ± 0.0043  Δbaseline=-0.0018


## CELDA 7 — EXP2/3/4: Vitales continuos (temperature, heartrate, dbp)
*Requiere CELDAS 2 y 3.*

In [7]:
# =========================================================
# EXP2/3/4 — Vitales continuos sobre 46 features
# =========================================================
# EXP2: 46 + ccs_mean_acuity + temperature  → descartado (leakage ccs)
# EXP3: 46 + temperature_valor              → F1=0.4905
# EXP4: 46 + temperature + heartrate + dbp  → F1=0.4931 ✅
# =========================================================

# Medianas sobre train únicamente — sin leakage
temp_mediana = df_train["temperature"].median()
hr_mediana   = df_train["heartrate"].median()
dbp_mediana  = df_train["dbp"].median()

feats_temp_train = df_train["temperature"].fillna(temp_mediana).rename("temperature_valor").reset_index(drop=True)
feats_temp_test  = df_test["temperature"].fillna(temp_mediana).rename("temperature_valor").reset_index(drop=True)
feats_hr_train   = df_train["heartrate"].fillna(hr_mediana).rename("heartrate_valor").reset_index(drop=True)
feats_hr_test    = df_test["heartrate"].fillna(hr_mediana).rename("heartrate_valor").reset_index(drop=True)
feats_dbp_train  = df_train["dbp"].fillna(dbp_mediana).rename("dbp_valor").reset_index(drop=True)
feats_dbp_test   = df_test["dbp"].fillna(dbp_mediana).rename("dbp_valor").reset_index(drop=True)

X_tr_exp4 = pd.concat([X_tr_sel.reset_index(drop=True), feats_temp_train, feats_hr_train, feats_dbp_train], axis=1)
X_te_exp4 = pd.concat([X_te_sel.reset_index(drop=True), feats_temp_test,  feats_hr_test,  feats_dbp_test],  axis=1)

scores_exp4, n_trees_exp4 = [], []
lgbm_exp4 = LGBMClassifier(**params_modelo)
print(f"CV EXP4 ({X_tr_exp4.shape[1]} features)...")
for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_exp4, y_train, groups=groups_train)):
    lgbm_exp4.fit(
        X_tr_exp4.iloc[idx_tr], y_train.iloc[idx_tr]-1,
        sample_weight=sample_weights_train[idx_tr],
        eval_set=[(X_tr_exp4.iloc[idx_val], y_train.iloc[idx_val]-1)],
        eval_sample_weight=[sample_weights_train[idx_val]],
        callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(period=-1)],
    )
    s = macro_f1(y_train.iloc[idx_val], lgbm_exp4.predict(X_tr_exp4.iloc[idx_val])+1)
    scores_exp4.append(s); n_trees_exp4.append(lgbm_exp4.best_iteration_)
    print(f"  Fold {fold+1}: {s:.4f}  |  árboles: {lgbm_exp4.best_iteration_}")

f1_exp4   = np.mean(scores_exp4)
ganancia4 = f1_exp4 - f1_baseline
print(f"\n  EXP4 (49f): {f1_exp4:.4f} ± {np.std(scores_exp4):.4f}  Δbaseline={ganancia4:+.4f}")

if ganancia4 > 0.003:
    X_tr_exp4.to_parquet(NOTEBOOK_DIR / "X_train_exp4.parquet", index=False)
    X_te_exp4.to_parquet(NOTEBOOK_DIR / "X_test_exp4.parquet",  index=False)
    config_exp4 = {
        "features_originales": FEATURES_SELECCIONADAS,
        "features_nuevas": ["temperature_valor","heartrate_valor","dbp_valor"],
        "todas_features": list(X_tr_exp4.columns),
        "n_features": X_tr_exp4.shape[1],
        "macro_f1_baseline": f1_baseline, "macro_f1_exp4": round(f1_exp4,6),
        "ganancia_vs_baseline": round(ganancia4,6),
        "imputaciones": {"temperature": float(temp_mediana), "heartrate": float(hr_mediana), "dbp": float(dbp_mediana)},
    }
    with open(NOTEBOOK_DIR / "feature_config_exp4.json","w",encoding="utf-8") as f:
        json.dump(config_exp4, f, indent=2, ensure_ascii=False)
    print(f"  ✅ Exportados exp4 → {NOTEBOOK_DIR}")


CV EXP4 (49 features)...
  Fold 1: 0.4944  |  árboles: 408
  Fold 2: 0.4847  |  árboles: 386
  Fold 3: 0.5019  |  árboles: 436
  Fold 4: 0.4926  |  árboles: 381
  Fold 5: 0.4918  |  árboles: 411

  EXP4 (49f): 0.4931 ± 0.0055  Δbaseline=+0.0060
  ✅ Exportados exp4 → c:\Users\CARLOS\triaje-ia-tfg\notebooks\3_modeling


## CELDA 8 — EXP5: Kitchen sink + ablación por grupo
*Requiere CELDAS 2, 4 y 7 (X_tr_exp4 en memoria).*

In [8]:
# =========================================================
# EXP5 — Kitchen sink: exp4 + 38 features descartadas
# =========================================================

TODAS_DESCARTADAS = [f for g in GRUPOS_DESCARTADAS.values() for f in g if f in df_88_train.columns]

# Validación univariante
print("Validación univariante por grupo...")
features_con_senal = []
for grupo, features in GRUPOS_DESCARTADAS.items():
    feats_g = [f for f in features if f in df_88_train.columns]
    print(f"\n  [{grupo}]")
    for f in feats_g:
        col = df_88_train[f].fillna(0)
        if col.std() > 0:
            grupos_k = [col[y_train.values==k].values for k in sorted(y_train.unique())]
            H, _ = kruskal(*grupos_k)
            flag = "✅" if H>50 else ("⚠️ " if H>10 else "🚨")
            print(f"    {flag} {f:<35} H={H:>8.1f}")
            if H > 10: features_con_senal.append(f)

feats_ks_tr = df_88_train[features_con_senal].fillna(0).reset_index(drop=True).astype("float32")
feats_ks_te = df_88_test[features_con_senal].fillna(0).reset_index(drop=True).astype("float32")
X_tr_exp5   = pd.concat([X_tr_exp4.reset_index(drop=True), feats_ks_tr], axis=1)
X_te_exp5   = pd.concat([X_te_exp4.reset_index(drop=True), feats_ks_te], axis=1)

scores_exp5, n_trees_exp5 = [], []
lgbm_exp5 = LGBMClassifier(**params_modelo)
print(f"\nCV kitchen sink ({X_tr_exp5.shape[1]} features)...")
for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_exp5, y_train, groups=groups_train)):
    lgbm_exp5.fit(
        X_tr_exp5.iloc[idx_tr], y_train.iloc[idx_tr]-1,
        sample_weight=sample_weights_train[idx_tr],
        eval_set=[(X_tr_exp5.iloc[idx_val], y_train.iloc[idx_val]-1)],
        eval_sample_weight=[sample_weights_train[idx_val]],
        callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(period=-1)],
    )
    s = macro_f1(y_train.iloc[idx_val], lgbm_exp5.predict(X_tr_exp5.iloc[idx_val])+1)
    scores_exp5.append(s); n_trees_exp5.append(lgbm_exp5.best_iteration_)
    print(f"  Fold {fold+1}: {s:.4f}  |  árboles: {lgbm_exp5.best_iteration_}")

f1_exp5    = np.mean(scores_exp5)
delta_exp4 = f1_exp5 - f1_exp4
print(f"\n  Kitchen sink ({X_tr_exp5.shape[1]}f): {f1_exp5:.4f} ± {np.std(scores_exp5):.4f}  Δexp4={delta_exp4:+.4f}")

# Ablación por grupo
if delta_exp4 > 0.001:
    print("\n  Ablación por grupo...")
    resultados_ablacion = {}
    for grupo, features in GRUPOS_DESCARTADAS.items():
        feats_g = [f for f in features if f in features_con_senal]
        if not feats_g: continue
        X_tr_abl = pd.concat([X_tr_exp4.reset_index(drop=True),
                               df_88_train[feats_g].fillna(0).reset_index(drop=True).astype("float32")], axis=1)
        scores_abl = []
        lgbm_abl = LGBMClassifier(**params_modelo)
        for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_abl, y_train, groups=groups_train)):
            lgbm_abl.fit(
                X_tr_abl.iloc[idx_tr], y_train.iloc[idx_tr]-1,
                sample_weight=sample_weights_train[idx_tr],
                eval_set=[(X_tr_abl.iloc[idx_val], y_train.iloc[idx_val]-1)],
                eval_sample_weight=[sample_weights_train[idx_val]],
                callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(period=-1)],
            )
            scores_abl.append(macro_f1(y_train.iloc[idx_val], lgbm_abl.predict(X_tr_abl.iloc[idx_val])+1))
        f1_abl = np.mean(scores_abl)
        delta  = f1_abl - f1_exp4
        flag   = "✅" if delta>0.001 else ("⚠️ " if delta>=0 else "❌")
        resultados_ablacion[grupo] = {"f1": f1_abl, "delta": delta, "features": feats_g}
        print(f"  {flag} {grupo:<20} F1={f1_abl:.4f}  Δexp4={delta:+.4f}  n={len(feats_g)}")
    print("\n  Ranking:")
    for g, v in sorted(resultados_ablacion.items(), key=lambda x: -x[1]["f1"]):
        print(f"    {g:<20} {v['f1']:.4f}  {v['delta']:+.4f}")


Validación univariante por grupo...

  [vitales_flags]
    ✅ fiebre                              H=   815.3
    ✅ hipotension                         H= 24461.6
    ✅ hipertension_severa                 H=   883.6
    ✅ qsofa_positivo                      H=  2122.4
    ✅ shock_index_severo                  H=  8137.0
    ✅ anciano                             H= 13476.7
    ✅ anciano_mayor                       H=  9004.9

  [visitas]
    ✅ n_visitas_previas                   H=   729.8
    ✅ visitas_ultimo_mes                  H=   389.3
    ✅ visitas_ultimo_año                  H=   719.6
    ✅ dias_desde_ultima_visita            H=   401.2
    ✅ primera_visita                      H=   548.6
    ✅ frecuentador                        H=   393.5

  [cc_extra]
    ✅ cc_cefalea                          H=   706.3
    ✅ cc_intoxicacion                     H=   696.8
    ✅ cc_infeccioso                       H=  1052.5
    ✅ cc_hemorragia_activa                H=  1308.4

  [hx_extra]
   

## CELDA 9 — EXP5b: Ablación acumulativa (punto de corte óptimo)
*Requiere CELDA 8 ejecutada. Exporta X_train_exp5b.parquet.*

In [9]:
# =========================================================
# EXP5B — Ablación acumulativa greedy
# =========================================================
# Orden por ranking de ablación individual:
#   1. visitas       +0.016
#   2. cc_extra      +0.008
#   3. hx_extra      +0.003
#   4. llegada_extra +0.001  (pierde al combinarse — esperado ❌)
#   5. med_extra     +0.000  (ruido neto)
# vitales_flags excluido (Δ=-0.0004)
# =========================================================

GRUPOS_ACUMULATIVOS = [
    ("visitas",       ["n_visitas_previas","visitas_ultimo_mes","visitas_ultimo_año",
                       "dias_desde_ultima_visita","primera_visita","frecuentador"]),
    ("cc_extra",      ["cc_cefalea","cc_intoxicacion","cc_infeccioso","cc_hemorragia_activa"]),
    ("hx_extra",      ["hx_respiratorio","hx_neuro","hx_psiquiatrico","hx_abuso_sustancias",
                       "hx_digestivo","hx_metabolico_renal","hx_infeccioso","hx_trauma_muscular"]),
    ("llegada_extra", ["llegada_autonoma"]),
    ("med_extra",     ["med_anticoagulante","med_antidiabetico_oral","med_corticoide_sistemico",
                       "med_benzodiacepina","med_diuretico_tiazida","med_respiratorio_inhalado",
                       "med_inmunosupresor","med_digoxina","med_antiaritmico",
                       "riesgo_depresion_resp","polifarmacia","hiperpolifarmacia"]),
]

print("=" * 65)
print("  ABLACIÓN ACUMULATIVA — añadiendo grupos en orden de ranking")
print("=" * 65)
print(f"  Base: exp4 ({X_tr_exp4.shape[1]} features, F1={f1_exp4:.4f})\n")

resultados_acum   = {}
features_acum     = []
f1_anterior       = f1_exp4

for nombre_grupo, features_grupo in GRUPOS_ACUMULATIVOS:
    feats_v = [f for f in features_grupo if f in df_88_train.columns]
    features_acum = features_acum + feats_v
    nuevas_tr = df_88_train[features_acum].fillna(0).reset_index(drop=True).astype("float32")
    X_tr_acum = pd.concat([X_tr_exp4.reset_index(drop=True), nuevas_tr], axis=1)

    scores_acum, n_trees_acum = [], []
    lgbm_acum = LGBMClassifier(**params_modelo)
    for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_acum, y_train, groups=groups_train)):
        lgbm_acum.fit(
            X_tr_acum.iloc[idx_tr], y_train.iloc[idx_tr]-1,
            sample_weight=sample_weights_train[idx_tr],
            eval_set=[(X_tr_acum.iloc[idx_val], y_train.iloc[idx_val]-1)],
            eval_sample_weight=[sample_weights_train[idx_val]],
            callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(period=-1)],
        )
        scores_acum.append(macro_f1(y_train.iloc[idx_val], lgbm_acum.predict(X_tr_acum.iloc[idx_val])+1))
        n_trees_acum.append(lgbm_acum.best_iteration_)

    f1_acum           = np.mean(scores_acum)
    delta_vs_exp4     = f1_acum - f1_exp4
    delta_vs_anterior = f1_acum - f1_anterior
    flag = "✅" if delta_vs_anterior>0.001 else ("⚠️ " if delta_vs_anterior>=0 else "❌")
    print(f"  {flag} +{nombre_grupo:<15} n={X_tr_acum.shape[1]:>3}f  F1={f1_acum:.4f}  "
          f"Δexp4={delta_vs_exp4:+.5f}  Δant={delta_vs_anterior:+.5f}  árboles={np.mean(n_trees_acum):.0f}")

    resultados_acum[nombre_grupo] = {
        "f1": f1_acum, "delta_exp4": delta_vs_exp4, "delta_anterior": delta_vs_anterior,
        "n_features": X_tr_acum.shape[1], "features_acum": features_acum.copy(),
        "n_arboles": int(np.mean(n_trees_acum)),
    }
    f1_anterior = f1_acum

# Punto de corte óptimo
f1_max_acum  = max(v["f1"] for v in resultados_acum.values())
mejor_acum   = max(resultados_acum, key=lambda x: resultados_acum[x]["f1"])
config_mejor = resultados_acum[mejor_acum]
feats_def    = config_mejor["features_acum"]
FEATURES_EXP5B = list(X_tr_exp4.columns) + feats_def

X_tr_exp5b = pd.concat([X_tr_exp4.reset_index(drop=True),
                         df_88_train[feats_def].fillna(0).reset_index(drop=True).astype("float32")], axis=1)
X_te_exp5b = pd.concat([X_te_exp4.reset_index(drop=True),
                         df_88_test[feats_def].fillna(0).reset_index(drop=True).astype("float32")],  axis=1)
f1_exp5b   = round(f1_max_acum, 6)

print(f"\n  Mejor: {mejor_acum} ({config_mejor['n_features']}f, F1={f1_max_acum:.4f})")
print(f"  Δbaseline={f1_max_acum-f1_baseline:+.4f}  |  Δexp4={f1_max_acum-f1_exp4:+.4f}")

if config_mejor["delta_exp4"] > 0.003:
    X_tr_exp5b.to_parquet(NOTEBOOK_DIR / "X_train_exp5b.parquet", index=False)
    X_te_exp5b.to_parquet(NOTEBOOK_DIR / "X_test_exp5b.parquet",  index=False)
    config_exp5b = {
        "features_exp4": list(X_tr_exp4.columns), "features_nuevas": feats_def,
        "todas_features": FEATURES_EXP5B, "n_features": len(FEATURES_EXP5B),
        "grupos_incorporados": [g for g in resultados_acum if resultados_acum[g]["delta_anterior"]>0.001],
        "grupos_descartados":  [g for g in resultados_acum if resultados_acum[g]["delta_anterior"]<=0.001],
        "macro_f1_baseline": f1_baseline, "macro_f1_exp4": f1_exp4,
        "macro_f1_exp5b": f1_exp5b,
        "ganancia_vs_baseline": round(f1_exp5b-f1_baseline,6),
        "ganancia_vs_exp4":     round(f1_exp5b-f1_exp4,6),
    }
    with open(NOTEBOOK_DIR / "feature_config_exp5b.json","w",encoding="utf-8") as f:
        json.dump(config_exp5b, f, indent=2, ensure_ascii=False)
    print(f"  ✅ Exportados exp5b → {NOTEBOOK_DIR}")
print("=" * 65)


  ABLACIÓN ACUMULATIVA — añadiendo grupos en orden de ranking
  Base: exp4 (49 features, F1=0.4931)

  ✅ +visitas         n= 55f  F1=0.5093  Δexp4=+0.01619  Δant=+0.01619  árboles=419
  ✅ +cc_extra        n= 59f  F1=0.5173  Δexp4=+0.02421  Δant=+0.00802  árboles=479
  ✅ +hx_extra        n= 67f  F1=0.5198  Δexp4=+0.02675  Δant=+0.00253  árboles=474
  ❌ +llegada_extra   n= 68f  F1=0.5177  Δexp4=+0.02466  Δant=-0.00209  árboles=473
  ✅ +med_extra       n= 80f  F1=0.5196  Δexp4=+0.02654  Δant=+0.00188  árboles=467

  Mejor: hx_extra (67f, F1=0.5198)
  Δbaseline=+0.0327  |  Δexp4=+0.0267
  ✅ Exportados exp5b → c:\Users\CARLOS\triaje-ia-tfg\notebooks\3_modeling


## CELDA 10 — Validación test holdout exp5b
*Entrena sobre train completo y evalúa en test independiente. Requiere CELDA 9 o CELDA 5.*

In [10]:
# =========================================================
# VALIDACIÓN FINAL EXP5B — Test holdout temporal
# =========================================================
# CV F1=0.5198 (desarrollo) — este es el resultado real.
# =========================================================

lgbm_final_5b = LGBMClassifier(**params_modelo)
lgbm_final_5b.fit(
    X_tr_exp5b, y_train - 1,
    sample_weight = sample_weights_train,
    callbacks     = [lgb.log_evaluation(period=-1)],
)

y_pred_test_5b = lgbm_final_5b.predict(X_te_exp5b) + 1
f1_test_5b     = macro_f1(y_test, y_pred_test_5b)

print("=" * 55)
print("  VALIDACIÓN FINAL — Test holdout temporal")
print("=" * 55)
print(f"  CV  Macro F1 (desarrollo) : {f1_exp5b:.4f}")
print(f"  Test Macro F1 (real)      : {f1_test_5b:.4f}")
print(f"  Diferencia CV vs Test     : {f1_test_5b - f1_exp5b:+.4f}")
print()
print(classification_report(y_test, y_pred_test_5b,
      target_names=["acuity1","acuity2","acuity3","acuity4","acuity5"]))
print("=" * 55)
# Resultado conocido: Test F1=0.5056, gap=-0.0142 (sesgo temporal normal)


  VALIDACIÓN FINAL — Test holdout temporal
  CV  Macro F1 (desarrollo) : 0.5198
  Test Macro F1 (real)      : 0.5056
  Diferencia CV vs Test     : -0.0142

              precision    recall  f1-score   support

     acuity1       0.71      0.67      0.69      4648
     acuity2       0.64      0.64      0.64     28149
     acuity3       0.73      0.69      0.71     45391
     acuity4       0.33      0.50      0.40      5264
     acuity5       0.16      0.06      0.09       168

    accuracy                           0.66     83620
   macro avg       0.51      0.51      0.51     83620
weighted avg       0.67      0.66      0.67     83620



## CELDA 11 — EXP-TH: Threshold Tuning (Nelder-Mead)
*Requiere CELDA 10 (lgbm_final_5b en memoria). Mejora esperada: +0.010–0.025 en test.*

In [11]:
# =========================================================
# EXP-TH — Threshold Tuning post-hoc (Nelder-Mead)
# =========================================================
# Genera OOF probas y optimiza 5 pesos (uno por clase)
# para maximizar Macro F1 sin reentrenar el modelo.
# =========================================================

# 1. OOF probas sobre exp5b
print("Generando OOF probas exp5b...\n")
oof_proba_5b = np.zeros((len(y_train), 5))
oof_preds_5b = np.zeros(len(y_train), dtype=int)
lgbm_th = LGBMClassifier(**params_modelo)

for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_exp5b, y_train, groups=groups_train)):
    lgbm_th.fit(
        X_tr_exp5b.iloc[idx_tr], y_train.iloc[idx_tr]-1,
        sample_weight=sample_weights_train[idx_tr],
        eval_set=[(X_tr_exp5b.iloc[idx_val], y_train.iloc[idx_val]-1)],
        eval_sample_weight=[sample_weights_train[idx_val]],
        callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(period=-1)],
    )
    oof_proba_5b[idx_val] = lgbm_th.predict_proba(X_tr_exp5b.iloc[idx_val])
    oof_preds_5b[idx_val] = np.argmax(oof_proba_5b[idx_val], axis=1)
    s = macro_f1(y_train.iloc[idx_val]-1, oof_preds_5b[idx_val])
    print(f"  Fold {fold+1}: {s:.4f}  |  árboles: {lgbm_th.best_iteration_}")

f1_oof_5b = macro_f1(y_train.values-1, oof_preds_5b)
print(f"\n  OOF F1 sin thresholds: {f1_oof_5b:.4f}  (check vs CV: {f1_exp5b:.4f})")

# 2. Optimización Nelder-Mead
result = minimize(
    objetivo_th, x0=np.ones(5),
    args=(oof_proba_5b, y_train.values-1),
    method="Nelder-Mead",
    options={"maxiter": 10000, "xatol": 1e-6, "fatol": 1e-6, "adaptive": True},
)
w_opt         = result.x
f1_th_oof     = -result.fun
y_pred_th_oof = aplicar_thresholds(oof_proba_5b, w_opt)

print("\n" + "=" * 65)
print("  EXP-TH — Threshold Tuning (OOF, optimista)")
print("=" * 65)
print(f"  OOF F1 sin thresholds : {f1_oof_5b:.4f}")
print(f"  OOF F1 con thresholds : {f1_th_oof:.4f}  (Δ={f1_th_oof-f1_oof_5b:+.4f})")
print(f"\n  Pesos óptimos por clase:")
for i, w in enumerate(w_opt):
    n = y_train.value_counts().sort_index().iloc[i]
    print(f"    Clase {i+1} (acuity {i+1}, n={n:,}): w={w:.4f}")
print()
print(classification_report(y_train.values-1, y_pred_th_oof,
      target_names=["acuity1","acuity2","acuity3","acuity4","acuity5"]))

# 3. Validación real en test
proba_test_5b  = lgbm_final_5b.predict_proba(X_te_exp5b)
y_pred_th_test = aplicar_thresholds(proba_test_5b, w_opt) + 1
f1_th_test     = macro_f1(y_test, y_pred_th_test)

print("=" * 65)
print("  EXP-TH — Resultado real en test holdout")
print("=" * 65)
print(f"  Test F1 sin thresholds (exp5b) : {f1_test_5b:.4f}")
print(f"  Test F1 con thresholds         : {f1_th_test:.4f}  (Δ={f1_th_test-f1_test_5b:+.4f})")
print()
print(classification_report(y_test, y_pred_th_test,
      target_names=["acuity1","acuity2","acuity3","acuity4","acuity5"]))
print("=" * 65)


Generando OOF probas exp5b...

  Fold 1: 0.5187  |  árboles: 481
  Fold 2: 0.5116  |  árboles: 443
  Fold 3: 0.5255  |  árboles: 494
  Fold 4: 0.5211  |  árboles: 472
  Fold 5: 0.5222  |  árboles: 478

  OOF F1 sin thresholds: 0.5198  (check vs CV: 0.5198)

  EXP-TH — Threshold Tuning (OOF, optimista)
  OOF F1 sin thresholds : 0.5198
  OOF F1 con thresholds : 0.5224  (Δ=+0.0026)

  Pesos óptimos por clase:
    Clase 1 (acuity 1, n=19,371): w=0.7552
    Clase 2 (acuity 2, n=111,262): w=1.1384
    Clase 3 (acuity 3, n=179,675): w=1.1327
    Clase 4 (acuity 4, n=23,240): w=1.0282
    Clase 5 (acuity 5, n=932): w=0.9836

              precision    recall  f1-score   support

     acuity1       0.74      0.64      0.69     19371
     acuity2       0.63      0.64      0.64    111262
     acuity3       0.73      0.69      0.71    179675
     acuity4       0.35      0.51      0.41     23240
     acuity5       0.19      0.15      0.17       932

    accuracy                           0.66    33

## CELDA 12 — EXP-ST: Stacking LGB + XGB + CAT
*Requiere CELDA 11 (oof_proba_5b en memoria) y CELDA 10 (lgbm_final_5b).*

In [15]:
# =========================================================
# EXP-ST — Stacking (LGB + XGB + CAT → meta LogReg)
#           params propios validados en 06_modeling
# =========================================================

# ----- Params XGBoost (propios, validados en 06_modeling) -----
params_xgb = {
    "n_estimators":          1000,          # techo; early stopping decide
    "objective":             "multi:softprob",
    "num_class":             5,
    "eval_metric":           "mlogloss",
    "learning_rate":         0.03,          # lr bajo → más árboles, mejor generalización
    "max_depth":             6,
    "min_child_weight":      10,            # protege clase 5 (932 casos)
    "subsample":             0.8,
    "colsample_bytree":      0.8,
    "reg_alpha":             0.1,           # L1
    "reg_lambda":            1.0,           # L2
    "early_stopping_rounds": 50,
    "random_state":          42,
    "n_jobs":                -1,
    "verbosity":             0,
}

# ----- Params CatBoost (propios, validados en 06_modeling) -----
params_cat = {
    "iterations":            3000,          # techo; early stopping decide
    "learning_rate":         0.03,
    "depth":                 6,
    "bootstrap_type":        "Bernoulli",
    "subsample":             0.8,
    "colsample_bylevel":     0.8,
    "l2_leaf_reg":           1.0,
    "early_stopping_rounds": 50,
    "random_seed":           42,
    "thread_count":          -1,
    "verbose":               0,
}

# ----- OOF XGB -----
print("OOF XGB...")
oof_proba_xgb  = np.zeros((len(y_train), 5))
n_trees_xgb_st = []

for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_exp5b, y_train, groups=groups_train)):
    m = xgb.XGBClassifier(**params_xgb)
    m.fit(
        X_tr_exp5b.iloc[idx_tr],  y_train.iloc[idx_tr] - 1,
        sample_weight          = sample_weights_train[idx_tr],
        eval_set               = [(X_tr_exp5b.iloc[idx_val], y_train.iloc[idx_val] - 1)],
        sample_weight_eval_set = [sample_weights_train[idx_val]],   # early stopping ponderado
        verbose                = False,
    )
    oof_proba_xgb[idx_val] = m.predict_proba(X_tr_exp5b.iloc[idx_val])
    n_trees_xgb_st.append(m.best_iteration)
    f1_fold = macro_f1(y_train.iloc[idx_val] - 1, np.argmax(oof_proba_xgb[idx_val], axis=1))
    print(f"  Fold {fold+1}: {f1_fold:.4f}  |  árboles: {m.best_iteration}")

f1_xgb_oof = macro_f1(y_train.values - 1, np.argmax(oof_proba_xgb, axis=1))
print(f"  OOF F1 XGB: {f1_xgb_oof:.4f}\n")

# ----- OOF CatBoost -----
print("OOF CatBoost...")
oof_proba_cat  = np.zeros((len(y_train), 5))
n_trees_cat_st = []

for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_exp5b, y_train, groups=groups_train)):
    train_pool = cb.Pool(
        X_tr_exp5b.iloc[idx_tr],  y_train.iloc[idx_tr] - 1,
        weight = sample_weights_train[idx_tr],
    )
    val_pool = cb.Pool(
        X_tr_exp5b.iloc[idx_val], y_train.iloc[idx_val] - 1,
        weight = sample_weights_train[idx_val],                     # early stopping ponderado
    )
    m = cb.CatBoostClassifier(**params_cat)
    m.fit(train_pool, eval_set=val_pool)
    oof_proba_cat[idx_val] = m.predict_proba(X_tr_exp5b.iloc[idx_val])
    n_trees_cat_st.append(m.best_iteration_)
    f1_fold = macro_f1(y_train.iloc[idx_val] - 1, np.argmax(oof_proba_cat[idx_val], axis=1))
    print(f"  Fold {fold+1}: {f1_fold:.4f}  |  árboles: {m.best_iteration_}")

f1_cat_oof = macro_f1(y_train.values - 1, np.argmax(oof_proba_cat, axis=1))
print(f"  OOF F1 CAT: {f1_cat_oof:.4f}\n")

# ----- Meta-modelo -----
X_meta_train = np.hstack([oof_proba_5b, oof_proba_xgb, oof_proba_cat])
meta = LogisticRegression(max_iter=2000, C=0.5, random_state=42)
meta.fit(X_meta_train, y_train.values - 1)

# ----- Modelos finales sobre train completo -----
# n_estimators fijado al promedio de folds: sin eval_set no hay early stopping
print("Entrenando modelos finales sobre train completo...")
print(f"  XGB: {int(np.mean(n_trees_xgb_st))} árboles  |  CAT: {int(np.mean(n_trees_cat_st))} árboles")

params_xgb_final = {**params_xgb,
                    "n_estimators": int(np.mean(n_trees_xgb_st)),
                    "early_stopping_rounds": None}
m_xgb_final = xgb.XGBClassifier(**params_xgb_final)
m_xgb_final.fit(
    X_tr_exp5b, y_train - 1,
    sample_weight = sample_weights_train,
    verbose       = False,
)

params_cat_final = {**params_cat,
                    "iterations": int(np.mean(n_trees_cat_st)),
                    "early_stopping_rounds": None}
train_pool_full = cb.Pool(X_tr_exp5b, y_train - 1, weight=sample_weights_train)
m_cat_final = cb.CatBoostClassifier(**params_cat_final)
m_cat_final.fit(train_pool_full)

# ----- Test holdout -----
X_meta_test = np.hstack([
    lgbm_final_5b.predict_proba(X_te_exp5b),
    m_xgb_final.predict_proba(X_te_exp5b),
    m_cat_final.predict_proba(X_te_exp5b),
])
y_pred_stack_test = meta.predict(X_meta_test) + 1
f1_stack_test     = macro_f1(y_test, y_pred_stack_test)

print("\n" + "=" * 65)
print("  EXP-ST — Resultado en test holdout")
print("=" * 65)
print(f"  Test F1 LGB solo (exp5b) : {f1_test_5b:.4f}")
print(f"  OOF  F1 LGB              : {f1_oof_5b:.4f}")
print(f"  OOF  F1 XGB              : {f1_xgb_oof:.4f}")
print(f"  OOF  F1 CAT              : {f1_cat_oof:.4f}")
print(f"  Test F1 stacking         : {f1_stack_test:.4f}  (Δ={f1_stack_test-f1_test_5b:+.4f})")
print()
print(classification_report(y_test, y_pred_stack_test,
      target_names=["acuity1","acuity2","acuity3","acuity4","acuity5"]))
print("=" * 65)


OOF XGB...


KeyboardInterrupt: 

## CELDA 13 — EXP-CA: Cascada urgente / no urgente
*Requiere CELDAS 2 y 9. Evalúa en CV (OOF).*

In [16]:
# =========================================================
# EXP-CA — Modelo en cascada binaria
# =========================================================
# Etapa 1 : Binario {1-3} vs {4-5}
# Etapa 2A: Entre 1, 2, 3
# Etapa 2B: Entre 4, 5
# =========================================================

params_e1  = params_para_etapa(params_modelo, "binary",     n_clases=2, n_estimadores=1200)
params_e2a = params_para_etapa(params_modelo, "multiclass", n_clases=3, n_estimadores=1200)
params_e2b = params_para_etapa(params_modelo, "binary",     n_clases=2, n_estimadores=1200)

print("=" * 65)
print("  EXP-CA — Cascada urgente / no urgente")
print("=" * 65)
print(f"  Urgente    (1-3): {(y_train<=3).sum():,}  ({(y_train<=3).mean()*100:.1f}%)")
print(f"  No urgente (4-5): {(y_train>=4).sum():,}  ({(y_train>=4).mean()*100:.1f}%)\n")

oof_cascada = np.zeros(len(y_train), dtype=int)
scores_ca   = []

for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_exp5b, y_train, groups=groups_train)):
    X_ftr  = X_tr_exp5b.iloc[idx_tr];  X_fval = X_tr_exp5b.iloc[idx_val]
    y_ftr  = y_train.iloc[idx_tr];     y_fval = y_train.iloc[idx_val]
    sw_ftr  = sample_weights_train[idx_tr]
    sw_fval = sample_weights_train[idx_val]

    # ── Etapa 1: urgente (1-3=1) vs no urgente (4-5=0) ──────
    m1 = LGBMClassifier(**params_e1)
    m1.fit(
        X_ftr, (y_ftr <= 3).astype(int),
        sample_weight = sw_ftr,
        eval_set      = [(X_fval, (y_fval <= 3).astype(int))],
        callbacks     = [lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(period=-1)],
    )
    pred_bin = m1.predict(X_fval)

    idx_val_arr   = np.array(idx_val)
    idx_urg_val   = idx_val_arr[pred_bin == 1]
    idx_nourg_val = idx_val_arr[pred_bin == 0]

    # ── Etapa 2A: entre 1, 2, 3 ──────────────────────────────
    mask_urg_tr  = (y_ftr <= 3).values
    mask_urg_val = (y_fval <= 3).values

    m2a = LGBMClassifier(**params_e2a)
    m2a.fit(
        X_ftr[mask_urg_tr], y_ftr.values[mask_urg_tr] - 1,
        sample_weight      = sw_ftr[mask_urg_tr],
        eval_set           = [(X_fval[mask_urg_val], y_fval.values[mask_urg_val] - 1)],
        eval_sample_weight = [sw_fval[mask_urg_val]],
        callbacks          = [lgb.early_stopping(50, verbose=False),
                              lgb.log_evaluation(period=-1)],
    )

    # ── Etapa 2B: entre 4 (=0) y 5 (=1) ─────────────────────
    mask_nourg_tr = (y_ftr >= 4).values

    m2b = LGBMClassifier(**params_e2b)
    m2b.fit(
        X_ftr[mask_nourg_tr],
        np.where(y_ftr.values[mask_nourg_tr] == 4, 0, 1),
        sample_weight = sw_ftr[mask_nourg_tr],
        callbacks     = [lgb.log_evaluation(period=-1)],
    )

    if len(idx_urg_val) > 0:
        oof_cascada[idx_urg_val]   = m2a.predict(X_tr_exp5b.iloc[idx_urg_val]) + 1
    if len(idx_nourg_val) > 0:
        oof_cascada[idx_nourg_val] = m2b.predict(X_tr_exp5b.iloc[idx_nourg_val]) + 4

    s = macro_f1(y_fval, oof_cascada[idx_val])
    scores_ca.append(s)
    print(f"  Fold {fold+1}: {s:.4f}  "
          f"(urg={len(idx_urg_val):,}  nourg={len(idx_nourg_val):,}  "
          f"árboles e1={m1.best_iteration_} e2a={m2a.best_iteration_})")

f1_ca = np.mean(scores_ca)
print("\n" + "=" * 65)
print(f"  EXP-CA CV Macro F1 : {f1_ca:.4f}  (Δexp5b={f1_ca - f1_exp5b:+.4f})")
print(classification_report(y_train, oof_cascada,
      target_names=["acuity1","acuity2","acuity3","acuity4","acuity5"]))
print("=" * 65)

NameError: name 'params_para_etapa' is not defined

## CELDA 14 — Resumen comparativo global
*Consolida todos los resultados. Requiere haber ejecutado las celdas de cada experimento.*

In [17]:
# =========================================================
# RESUMEN GLOBAL — Todos los experimentos
# =========================================================

resultados = {
    "Baseline      (46f)": {"test": f1_baseline,    "cv": None},
    "Exp4          (49f)": {"test": f1_exp4,         "cv": None},
    "Exp5b         (67f)": {"test": f1_test_5b,      "cv": f1_exp5b},
    "Exp5b + TH    (67f)": {"test": f1_th_test,      "cv": f1_th_oof},
    "Exp5b + Stack (67f)": {"test": f1_stack_test,   "cv": None},
    "Exp5b + Casc  (67f)": {"test": None,            "cv": f1_ca},
}

print("=" * 72)
print("  RESUMEN GLOBAL DE EXPERIMENTOS")
print(f"  {'Experimento':<26} {'Test F1':>10} {'CV F1':>10} {'Δ baseline':>12}")
print("=" * 72)
for nombre, v in resultados.items():
    t_str = f"{v['test']:.4f}" if v["test"] is not None else "    —   "
    c_str = f"{v['cv']:.4f}"   if v["cv"]   is not None else "    —   "
    ref   = v["test"] if v["test"] is not None else v["cv"]
    d_str = f"{ref - f1_baseline:+.4f}" if ref is not None else "    —   "
    print(f"  {nombre:<26} {t_str:>10} {c_str:>10} {d_str:>12}")

print("=" * 72)
tests = {k: v["test"] for k, v in resultados.items() if v["test"] is not None}
mejor = max(tests, key=tests.get)
print(f"\n  → Mejor resultado en test : {mejor}  ({tests[mejor]:.4f})")
print(f"  → ¿Supera 0.52?  {'✅ SÍ' if tests[mejor]>=0.52 else '❌ NO'}")
print(f"  → ¿Supera 0.53?  {'✅ SÍ' if tests[mejor]>=0.53 else '❌ NO — combinar TH + Stack'}")


NameError: name 'f1_ca' is not defined

In [ ]:
# =========================================================
# CELDA 15 — Resumen global de experimentos
# =========================================================

resumen = [
    {"Experimento": "Baseline tuned      (46f)", "CV F1": 0.4871, "Test F1": None,         "Δ baseline CV": 0.0},
    {"Experimento": "Exp5b               (67f)", "CV F1": 0.5198, "Test F1": 0.5056,        "Δ baseline CV": 0.5198 - 0.4871},
    {"Experimento": "Exp5b + Thresholds  (67f)", "CV F1": 0.5224, "Test F1": 0.5092,        "Δ baseline CV": 0.5224 - 0.4871},
    {"Experimento": "Exp5b + Stacking    (67f)", "CV F1": None,   "Test F1": 0.4686,        "Δ baseline CV": None},
    {"Experimento": "Exp5b + Cascada     (67f)", "CV F1": 0.5145, "Test F1": None,          "Δ baseline CV": 0.5145 - 0.4871},
]

print("=" * 72)
print("  RESUMEN GLOBAL DE EXPERIMENTOS")
print(f"  {'Experimento':<30} {'CV F1':>8} {'Test F1':>9} {'Δ baseline':>11}")
print("=" * 72)
for r in resumen:
    cv_str   = f"{r['CV F1']:.4f}"   if r["CV F1"]   is not None else "    —   "
    test_str = f"{r['Test F1']:.4f}" if r["Test F1"] is not None else "    —   "
    delta_str= f"{r['Δ baseline CV']:+.4f}" if r["Δ baseline CV"] is not None else "    —   "
    print(f"  {r['Experimento']:<30} {cv_str:>8} {test_str:>9} {delta_str:>11}")
print("=" * 72)

print("""
  MODELO DEFINITIVO: Exp5b + Threshold Tuning
  ─────────────────────────────────────────────────────────────────
  · Features  : 67 (49 exp4 + 18 del grupo visitas + cc_extra + hx_extra)
  · CV F1     : 0.5224  (+0.0353 sobre baseline)
  · Test F1   : 0.5092  (+0.0221 sobre baseline estimado)
  · Gap CV→Test: -0.0132  (shift temporal esperado en datos hospitalarios)

  EXPERIMENTOS DESCARTADOS
  ─────────────────────────────────────────────────────────────────
  · Stacking (LGB+XGB+CAT → LogReg):
      OOF competitivo (~0.518 los tres modelos) pero meta-modelo
      colapsa acuity4/5 en test → Test F1 0.4686 (Δ=-0.037).
      Causa: correlación alta entre base learners + shift temporal.

  · Cascada binaria (urgente/no urgente):
      Arquitectura estructuralmente limitada con 92.8% urgente.
      Etapa 1 toca techo (1200 árboles) y propaga errores sin
      posibilidad de corrección → CV F1 0.5145 (Δ=-0.0053 vs exp5b).
""")

print("=" * 72)
print(f"  ¿Supera 0.50 en test?  {'✅ SÍ' if 0.5092 >= 0.50 else '❌ NO'}")
print(f"  ¿Supera 0.51 en test?  {'✅ SÍ' if 0.5092 >= 0.51 else '❌ NO'}")
print(f"  ¿Supera 0.52 en test?  {'✅ SÍ' if 0.5092 >= 0.52 else '❌ NO'}")
print("=" * 72)

  RESUMEN GLOBAL DE EXPERIMENTOS
  Experimento                       CV F1   Test F1  Δ baseline
  Baseline tuned      (46f)        0.4871      —        +0.0000
  Exp5b               (67f)        0.5198    0.5056     +0.0327
  Exp5b + Thresholds  (67f)        0.5224    0.5092     +0.0353
  Exp5b + Stacking    (67f)          —       0.4686        —   
  Exp5b + Cascada     (67f)        0.5145      —        +0.0274

  MODELO DEFINITIVO: Exp5b + Threshold Tuning
  ─────────────────────────────────────────────────────────────────
  · Features  : 67 (49 exp4 + 18 del grupo visitas + cc_extra + hx_extra)
  · CV F1     : 0.5224  (+0.0353 sobre baseline)
  · Test F1   : 0.5092  (+0.0221 sobre baseline estimado)
  · Gap CV→Test: -0.0132  (shift temporal esperado en datos hospitalarios)

  EXPERIMENTOS DESCARTADOS
  ─────────────────────────────────────────────────────────────────
  · Stacking (LGB+XGB+CAT → LogReg):
      OOF competitivo (~0.518 los tres modelos) pero meta-modelo
      colapsa 

Exp5b + Threshold Tuning, sin duda.
Es el mejor en todas las dimensiones que importan:

CV F1 más alto: 0.5224 (el threshold tuning añade +0.0026 sobre exp5b puro)
Test F1 más alto: 0.5092 (el único experimento que mejora exp5b en test)
Gap CV→Test contenido: −0.0132, consistente con el shift temporal esperado
Coste computacional cero: los thresholds se calculan una vez sobre las OOF probas ya generadas, no requiere reentrenar nada

Los otros dos los descartas con causa clara:

Stacking: competitivo en OOF pero se hunde −0.037 en test por correlación entre modelos y shift temporal
Cascada: −0.0053 en CV, arquitectura limitada por el desbalanceo 92.8/7.2%

El pipeline definitivo es: 67 features (exp5b) + LightGBM tuneado + pesos óptimos por clase.

In [ ]:
print(f"Inicio train : {df_train['intime'].min()}")
print(f"Fecha corte  : {fecha_corte}")
print(f"Rango total  : {(fecha_corte - df_train['intime'].min()).days} días")

Inicio train : 2110-01-11 01:45:00
Fecha corte  : 2180-06-06 02:38:36
Rango total  : 25714 días


In [ ]:
# =========================================================
# EXP-TDW — Time-Decay Weights
# =========================================================
# Multiplica sample_weights_train por una exponencial
# decreciente basada en antigüedad respecto a fecha_corte.
# Los pacientes más recientes reciben más peso.
# =========================================================

import numpy as np

# Días de antigüedad de cada visita respecto a fecha_corte
dias_antiguedad = (fecha_corte - df_train["intime"]).dt.days.values
dias_antiguedad = np.clip(dias_antiguedad, 0, None)   # por si hay algún valor negativo residual

# Tres vidas medias a comparar (en días)
# - Suave  : últimos ~20% del rango tienen ventaja moderada
# - Medio  : equilibrio entre recencia y volumen histórico
# - Agresivo: da mucho más peso al último tercio del rango
half_lifes = {
    "suave"    : 20000,
    "medio"    : 10000,
    "agresivo" :  5000,
}

print("=" * 65)
print("  EXP-TDW — Time-Decay Weights")
print("=" * 65)
print(f"  Rango train: {dias_antiguedad.max()} días")
print(f"  Base weights — min: {sample_weights_train.min():.3f}  "
      f"max: {sample_weights_train.max():.3f}\n")

resultados_tdw = {}

for nombre, hl in half_lifes.items():

    # Peso temporal: 1.0 para el más reciente, decae hacia antiguo
    decay        = np.exp(-np.log(2) * dias_antiguedad / hl)
    sw_tdw       = sample_weights_train * decay

    # Renormalizar para que la suma sea igual que los pesos originales
    # (evita que el lr efectivo cambie por escala)
    sw_tdw = sw_tdw * (sample_weights_train.sum() / sw_tdw.sum())

    scores_tdw = []
    lgbm_tdw   = LGBMClassifier(**params_modelo)

    for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_exp5b, y_train, groups=groups_train)):
        lgbm_tdw.fit(
            X_tr_exp5b.iloc[idx_tr], y_train.iloc[idx_tr] - 1,
            sample_weight      = sw_tdw[idx_tr],
            eval_set           = [(X_tr_exp5b.iloc[idx_val], y_train.iloc[idx_val] - 1)],
            eval_sample_weight = [sw_tdw[idx_val]],
            callbacks          = [lgb.early_stopping(50, verbose=False),
                                  lgb.log_evaluation(period=-1)],
        )
        scores_tdw.append(macro_f1(
            y_train.iloc[idx_val],
            lgbm_tdw.predict(X_tr_exp5b.iloc[idx_val]) + 1
        ))

    f1_tdw = np.mean(scores_tdw)
    resultados_tdw[nombre] = {"f1": f1_tdw, "hl": hl, "sw": sw_tdw}
    delta  = f1_tdw - f1_exp5b
    flag   = "✅" if delta > 0.001 else ("⚠️ " if delta >= 0 else "❌")
    print(f"  {flag} half-life {hl:>6}d ({nombre:<9}) "
          f"CV F1: {f1_tdw:.4f}  Δexp5b: {delta:+.4f}")

print("\n" + "=" * 65)
mejor_tdw = max(resultados_tdw, key=lambda x: resultados_tdw[x]["f1"])
print(f"  Mejor configuración: {mejor_tdw} "
      f"(half-life {resultados_tdw[mejor_tdw]['hl']}d) "
      f"→ CV F1: {resultados_tdw[mejor_tdw]['f1']:.4f}")
print("=" * 65)

  EXP-TDW — Time-Decay Weights
  Rango train: 25714 días
  Base weights — min: 0.731  max: 10.148

  ❌ half-life  20000d (suave    ) CV F1: 0.5175  Δexp5b: -0.0023
  ❌ half-life  10000d (medio    ) CV F1: 0.5178  Δexp5b: -0.0021
  ❌ half-life   5000d (agresivo ) CV F1: 0.5177  Δexp5b: -0.0021

  Mejor configuración: medio (half-life 10000d) → CV F1: 0.5178


In [18]:
print(df_train["chiefcomplaint"].dtype)
print(df_train["chiefcomplaint"].isna().sum(), "nulos")
print(df_train["chiefcomplaint"].nunique(), "valores únicos")
print(df_train["chiefcomplaint"].value_counts().head(20))

object
15 nulos
50638 valores únicos
chiefcomplaint
Chest pain               8720
Abd pain                 8549
Dyspnea                  4645
SI                       4002
ABD PAIN                 3875
s/p Fall                 3854
ETOH                     3826
Wound eval               3597
Headache                 3097
Back pain                2509
Altered mental status    2094
Lower back pain          2080
CHEST PAIN               1995
MVC                      1958
N/V                      1947
S/P FALL                 1918
Syncope                  1844
BRBPR                    1802
Dizziness                1768
Fever                    1743
Name: count, dtype: int64


In [19]:
# =========================================================
# EXP-NLP — TF-IDF + SVD sobre chiefcomplaint
# =========================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline

# ---------------------------------------------------------
# 1. PREPROCESADO DE TEXTO
# ---------------------------------------------------------
def limpiar_cc(texto):
    if pd.isna(texto):
        return "desconocido"
    t = str(texto).lower().strip()
    # Abreviaciones médicas frecuentes → texto expandido
    abrevs = {
        r"\bsi\b":    "suicidal ideation",
        r"\betoh\b":  "alcohol intoxication",
        r"\bbrbpr\b": "rectal bleeding",
        r"\bmvc\b":   "motor vehicle accident",
        r"\bs/p\b":   "status post",
        r"\bn/v\b":   "nausea vomiting",
        r"\babd\b":   "abdominal",
        r"\bcp\b":    "chest pain",
        r"\bsob\b":   "shortness of breath",
        r"\baloc\b":  "altered level of consciousness",
        r"\bams\b":   "altered mental status",
        r"\bha\b":    "headache",
        r"\bhtn\b":   "hypertension",
        r"\bdm\b":    "diabetes",
        r"\bdka\b":   "diabetic ketoacidosis",
        r"\bpe\b":    "pulmonary embolism",
        r"\buti\b":   "urinary tract infection",
        r"\blle\b":   "left lower extremity",
        r"\brle\b":   "right lower extremity",
    }
    import re
    for patron, expansion in abrevs.items():
        t = re.sub(patron, expansion, t)
    # Eliminar caracteres no alfabéticos salvo espacios
    t = re.sub(r"[^a-z\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

cc_train = df_train["chiefcomplaint"].apply(limpiar_cc)
cc_test  = df_test["chiefcomplaint"].apply(limpiar_cc)

print(f"Ejemplos preprocesados:")
for orig, proc in zip(df_train["chiefcomplaint"].head(5), cc_train.head(5)):
    print(f"  '{orig}' → '{proc}'")

# ---------------------------------------------------------
# 2. TF-IDF + SVD — fit solo sobre train
# ---------------------------------------------------------
N_COMPONENTES = 15   # suficiente para capturar semántica sin ruido

tfidf = TfidfVectorizer(
    ngram_range     = (1, 2),    # unigramas + bigramas
    min_df          = 10,        # ignora términos que aparecen en <10 visitas
    max_df          = 0.95,      # ignora términos en >95% de visitas (stopwords médicas)
    sublinear_tf    = True,      # log(tf) — reduce peso de términos muy frecuentes
    strip_accents   = "unicode",
    analyzer        = "word",
)

svd = TruncatedSVD(n_components=N_COMPONENTES, random_state=42)

# Fit exclusivamente sobre train — sin leakage
tfidf_matrix_train = tfidf.fit_transform(cc_train)
svd.fit(tfidf_matrix_train)

cc_svd_train = svd.transform(tfidf_matrix_train)
cc_svd_test  = svd.transform(tfidf.transform(cc_test))

varianza_explicada = svd.explained_variance_ratio_.sum()
print(f"\nTF-IDF vocab: {len(tfidf.vocabulary_):,} términos")
print(f"SVD {N_COMPONENTES} componentes — varianza explicada: {varianza_explicada:.1%}")

# Convertir a DataFrame con nombres de columna
cols_svd = [f"cc_svd_{i:02d}" for i in range(N_COMPONENTES)]
df_cc_train = pd.DataFrame(cc_svd_train, columns=cols_svd,
                            index=X_tr_exp5b.index)
df_cc_test  = pd.DataFrame(cc_svd_test,  columns=cols_svd,
                            index=X_te_exp5b.index)

# ---------------------------------------------------------
# 3. VALIDACIÓN UNIVARIANTE — señal de cada componente
# ---------------------------------------------------------
print("\n" + "=" * 65)
print("  SEÑAL UNIVARIANTE — H Kruskal por componente SVD")
print("=" * 65)

componentes_con_senal = []
for col in cols_svd:
    grupos_k = [df_cc_train[col][y_train.values == k].values
                for k in sorted(y_train.unique())]
    H, _ = kruskal(*grupos_k)
    flag = "✅" if H > 50 else ("⚠️ " if H > 10 else "🚨")
    print(f"  {flag} {col}  H={H:>8.1f}")
    if H > 10:
        componentes_con_senal.append(col)

print(f"\n  Componentes con señal (H>10): "
      f"{len(componentes_con_senal)}/{N_COMPONENTES}")

# ---------------------------------------------------------
# 4. CV — exp5b + componentes SVD con señal
# ---------------------------------------------------------
X_tr_nlp = pd.concat([
    X_tr_exp5b.reset_index(drop=True),
    df_cc_train[componentes_con_senal].reset_index(drop=True),
], axis=1)

X_te_nlp = pd.concat([
    X_te_exp5b.reset_index(drop=True),
    df_cc_test[componentes_con_senal].reset_index(drop=True),
], axis=1)

print(f"\n  X_tr_nlp: {X_tr_nlp.shape}  "
      f"(67 exp5b + {len(componentes_con_senal)} componentes SVD)")

scores_nlp  = []
lgbm_nlp    = LGBMClassifier(**params_modelo)

print("\n" + "=" * 65)
print(f"  CV — Exp5b + NLP ({X_tr_nlp.shape[1]} features)")
print("=" * 65)

for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_nlp, y_train, groups=groups_train)):
    lgbm_nlp.fit(
        X_tr_nlp.iloc[idx_tr],  y_train.iloc[idx_tr] - 1,
        sample_weight      = sample_weights_train[idx_tr],
        eval_set           = [(X_tr_nlp.iloc[idx_val], y_train.iloc[idx_val] - 1)],
        eval_sample_weight = [sample_weights_train[idx_val]],
        callbacks          = [lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(period=-1)],
    )
    score = macro_f1(y_train.iloc[idx_val],
                     lgbm_nlp.predict(X_tr_nlp.iloc[idx_val]) + 1)
    scores_nlp.append(score)
    print(f"  Fold {fold+1} — Macro F1: {score:.4f}  "
          f"|  árboles: {lgbm_nlp.best_iteration_}")

f1_nlp   = np.mean(scores_nlp)
delta_nlp = f1_nlp - f1_exp5b
flag      = "✅" if delta_nlp > 0.001 else ("⚠️ " if delta_nlp >= 0 else "❌")
print(f"\n  {flag} Exp5b + NLP ({X_tr_nlp.shape[1]}f): "
      f"{f1_nlp:.4f} ± {np.std(scores_nlp):.4f}")
print(f"  Exp5b solo             (67f): {f1_exp5b:.4f}")
print(f"  Ganancia NLP               : {delta_nlp:+.4f}")
print("=" * 65)

Ejemplos preprocesados:
  'Abd pain, Abdominal distention' → 'abdominal pain abdominal distention'
  'Confusion, Hallucinations' → 'confusion hallucinations'
  'Altered mental status, B Pedal edema' → 'altered mental status b pedal edema'
  'LEFT CHEEK SWELLING, Abscess' → 'left cheek swelling abscess'
  'L CHEEK ABSCESS' → 'l cheek abscess'

TF-IDF vocab: 3,907 términos
SVD 15 componentes — varianza explicada: 30.6%

  SEÑAL UNIVARIANTE — H Kruskal por componente SVD
  ✅ cc_svd_00  H= 11661.4
  ✅ cc_svd_01  H=  4262.9
  ✅ cc_svd_02  H=  6412.3
  ✅ cc_svd_03  H= 23290.8
  ✅ cc_svd_04  H= 10624.6
  ✅ cc_svd_05  H=  9487.9
  ✅ cc_svd_06  H=  1594.0
  ✅ cc_svd_07  H=   707.0
  ✅ cc_svd_08  H=  1358.2
  ✅ cc_svd_09  H=  6331.9
  ✅ cc_svd_10  H=  2382.9
  ✅ cc_svd_11  H=  7332.6
  ✅ cc_svd_12  H= 20124.1
  ✅ cc_svd_13  H=  3725.9
  ✅ cc_svd_14  H=  4166.1

  Componentes con señal (H>10): 15/15

  X_tr_nlp: (334480, 82)  (67 exp5b + 15 componentes SVD)

  CV — Exp5b + NLP (82 features)


KeyboardInterrupt: 

In [ ]:
# =========================================================
# EXP-NLP — Validación test holdout + Threshold Tuning
# =========================================================

# ---------------------------------------------------------
# 1. MODELO FINAL — entrenar sobre train completo
# ---------------------------------------------------------
lgbm_final_nlp = LGBMClassifier(**params_modelo)
lgbm_final_nlp.fit(
    X_tr_nlp, y_train - 1,
    sample_weight = sample_weights_train,
    callbacks     = [lgb.log_evaluation(period=-1)],
)

# ---------------------------------------------------------
# 2. TEST HOLDOUT — resultado base sin thresholds
# ---------------------------------------------------------
y_pred_nlp_test = lgbm_final_nlp.predict(X_te_nlp) + 1
f1_nlp_test     = macro_f1(y_test, y_pred_nlp_test)

print("=" * 65)
print("  EXP-NLP — Resultado en test holdout")
print("=" * 65)
print(f"  CV  F1 (desarrollo) : {f1_nlp:.4f}")
print(f"  Test F1 sin TH      : {f1_nlp_test:.4f}  "
      f"(gap={f1_nlp_test - f1_nlp:+.4f})")
print(f"  Referencia exp5b+TH : 0.5092")
print(f"  Ganancia vs exp5b+TH: {f1_nlp_test - 0.5092:+.4f}")
print()
print(classification_report(y_test, y_pred_nlp_test,
      target_names=["acuity1","acuity2","acuity3","acuity4","acuity5"]))
print("=" * 65)

# ---------------------------------------------------------
# 3. OOF PROBAS — para threshold tuning
# ---------------------------------------------------------
print("\nGenerando OOF probas NLP...")
oof_proba_nlp = np.zeros((len(y_train), 5))
lgbm_th_nlp   = LGBMClassifier(**params_modelo)

for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_nlp, y_train, groups=groups_train)):
    lgbm_th_nlp.fit(
        X_tr_nlp.iloc[idx_tr],  y_train.iloc[idx_tr] - 1,
        sample_weight      = sample_weights_train[idx_tr],
        eval_set           = [(X_tr_nlp.iloc[idx_val], y_train.iloc[idx_val] - 1)],
        eval_sample_weight = [sample_weights_train[idx_val]],
        callbacks          = [lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(period=-1)],
    )
    oof_proba_nlp[idx_val] = lgbm_th_nlp.predict_proba(X_tr_nlp.iloc[idx_val])

f1_oof_nlp = macro_f1(y_train, np.argmax(oof_proba_nlp, axis=1) + 1)
print(f"  OOF F1 sin thresholds: {f1_oof_nlp:.4f}  "
      f"(check vs CV: {f1_nlp:.4f})")

# ---------------------------------------------------------
# 4. THRESHOLD TUNING sobre OOF NLP
# ---------------------------------------------------------
print("\n" + "=" * 65)
print("  EXP-NLP+TH — Threshold Tuning")
print("=" * 65)

result_th = minimize(
    objetivo_th,
    x0      = np.ones(5),
    args    = (oof_proba_nlp, y_train.values - 1),
    method  = "Nelder-Mead",
    options = {"maxiter": 10000, "xatol": 1e-5, "fatol": 1e-5},
)
w_opt_nlp   = result_th.x
f1_th_oof_nlp = macro_f1(y_train, aplicar_thresholds(oof_proba_nlp, w_opt_nlp) + 1)

print(f"  OOF F1 sin thresholds : {f1_oof_nlp:.4f}")
print(f"  OOF F1 con thresholds : {f1_th_oof_nlp:.4f}  "
      f"(Δ={f1_th_oof_nlp - f1_oof_nlp:+.4f})")
print(f"\n  Pesos óptimos por clase:")
for i, (w, n) in enumerate(zip(w_opt_nlp,
        [19371, 111262, 179675, 23240, 932])):
    print(f"    Clase {i+1} (acuity {i+1}, n={n:,}): w={w:.4f}")

# Test con thresholds
proba_test_nlp  = lgbm_final_nlp.predict_proba(X_te_nlp)
y_pred_nlp_th   = aplicar_thresholds(proba_test_nlp, w_opt_nlp) + 1
f1_nlp_th_test  = macro_f1(y_test, y_pred_nlp_th)

print("\n" + "=" * 65)
print("  EXP-NLP+TH — Resultado final en test holdout")
print("=" * 65)
print(f"  Test F1 NLP sin TH  : {f1_nlp_test:.4f}")
print(f"  Test F1 NLP con TH  : {f1_nlp_th_test:.4f}  "
      f"(Δ={f1_nlp_th_test - f1_nlp_test:+.4f})")
print(f"  Referencia exp5b+TH : 0.5092")
print(f"  Ganancia total      : {f1_nlp_th_test - 0.5092:+.4f}")
print()
print(classification_report(y_test, y_pred_nlp_th,
      target_names=["acuity1","acuity2","acuity3","acuity4","acuity5"]))
print("=" * 65)

  EXP-NLP — Resultado en test holdout
  CV  F1 (desarrollo) : 0.5722
  Test F1 sin TH      : 0.5613  (gap=-0.0109)
  Referencia exp5b+TH : 0.5092
  Ganancia vs exp5b+TH: +0.0521

              precision    recall  f1-score   support

     acuity1       0.69      0.70      0.70      4648
     acuity2       0.67      0.68      0.67     28149
     acuity3       0.77      0.73      0.75     45391
     acuity4       0.42      0.58      0.49      5264
     acuity5       0.29      0.15      0.20       168

    accuracy                           0.70     83620
   macro avg       0.57      0.57      0.56     83620
weighted avg       0.71      0.70      0.70     83620


Generando OOF probas NLP...
  OOF F1 sin thresholds: 0.5721  (check vs CV: 0.5722)

  EXP-NLP+TH — Threshold Tuning
  OOF F1 sin thresholds : 0.5721
  OOF F1 con thresholds : 0.5757  (Δ=+0.0036)

  Pesos óptimos por clase:
    Clase 1 (acuity 1, n=19,371): w=0.7957
    Clase 2 (acuity 2, n=111,262): w=1.1375
    Clase 3 (acuity 3

In [30]:
# =========================================================
# EXP-BERT — ClinicalBERT embeddings + SVD
# Reemplaza TF-IDF por embeddings densos médicos.
# Requiere: CELDAS 2, 3, 5 y haber ejecutado limpiar_cc
# (cc_train y cc_test ya deben estar en memoria)
# =========================================================
# Instalación (ejecutar una sola vez en terminal):
#   pip install transformers torch
#
# Nota GPU: tu RX 6700 XT requiere ROCm en Linux para GPU.
# En Windows se ejecuta en CPU — ~25-40 min para 334k filas.
# Hazlo una vez y guarda el parquet; no repetirás el proceso.
# =========================================================

import joblib
from sklearn.decomposition import TruncatedSVD
from scipy.stats import kruskal
import numpy as np, pandas as pd

BERT_MODEL    = "emilyalsentzer/Bio_ClinicalBERT"
N_COMPONENTES = 15   # mismo nº que TF-IDF para comparación justa

_BERT_TRAIN_PATH = DATA_PROCESSED / "bert_embeddings_train.parquet"
_BERT_TEST_PATH  = DATA_PROCESSED / "bert_embeddings_test.parquet"
_SVD_BERT_PATH   = DATA_PROCESSED / "svd_bert.joblib"

if _BERT_TRAIN_PATH.exists() and _BERT_TEST_PATH.exists() and _SVD_BERT_PATH.exists():
    # ── Carga rápida desde caché ──────────────────────────
    print("✅ Cargando embeddings BERT desde disco (caché)...")
    df_bert_train = pd.read_parquet(_BERT_TRAIN_PATH)
    df_bert_test  = pd.read_parquet(_BERT_TEST_PATH)
    svd_bert      = joblib.load(_SVD_BERT_PATH)
    emb_train_full = None   # no disponible desde caché (sólo necesario en barrido SVD)
    emb_test_full  = None
    cols_bert = list(df_bert_train.columns)
    print(f"  df_bert_train: {df_bert_train.shape}")
    print(f"  df_bert_test : {df_bert_test.shape}")
    print(f"  svd_bert     : {svd_bert.n_components} componentes  "
          f"(varianza: {svd_bert.explained_variance_ratio_.sum():.1%})")
else:
    # ── Generación completa (primera vez, ~25-40 min CPU) ─
    import torch
    from transformers import AutoTokenizer, AutoModel

    # 1. Cargar modelo
    print(f"Cargando {BERT_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
    bert      = AutoModel.from_pretrained(BERT_MODEL)
    bert.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    bert   = bert.to(device)
    print(f"  Dispositivo: {device}")

    # 2. Función de extracción por batches
    def get_cls_embeddings(texts: list, batch_size: int = 128) -> np.ndarray:
        """Devuelve vector CLS (768-dim) para cada texto."""
        embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc   = tokenizer(
                batch,
                padding    = True,
                truncation = True,
                max_length = 32,
                return_tensors = "pt",
            ).to(device)
            with torch.no_grad():
                out = bert(**enc)
            cls = out.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls)
            if i % 10000 == 0:
                print(f"  {i:>7}/{len(texts)} procesados...", flush=True)
        return np.vstack(embeddings)

    # 3. Extraer embeddings
    cc_train_list = cc_train.tolist()
    cc_test_list  = cc_test.tolist()

    print(f"\nGenerando embeddings train ({len(cc_train_list):,} filas)...")
    emb_train_full = get_cls_embeddings(cc_train_list)

    print(f"\nGenerando embeddings test ({len(cc_test_list):,} filas)...")
    emb_test_full  = get_cls_embeddings(cc_test_list)

    # 4. SVD — fit SOLO sobre train (sin leakage)
    print(f"\nAplicando SVD ({N_COMPONENTES} componentes)...")
    svd_bert = TruncatedSVD(n_components=N_COMPONENTES, random_state=42)
    bert_svd_train = svd_bert.fit_transform(emb_train_full)
    bert_svd_test  = svd_bert.transform(emb_test_full)
    print(f"  Varianza explicada: {svd_bert.explained_variance_ratio_.sum():.1%}")

    # 5. DataFrames con nombres de columna
    cols_bert = [f"bert_svd_{i:02d}" for i in range(N_COMPONENTES)]
    df_bert_train = pd.DataFrame(bert_svd_train, columns=cols_bert,
                                  index=X_tr_exp5b.index)
    df_bert_test  = pd.DataFrame(bert_svd_test,  columns=cols_bert,
                                  index=X_te_exp5b.index)

    # 6. Guardar para no repetir el proceso
    df_bert_train.to_parquet(_BERT_TRAIN_PATH, index=False)
    df_bert_test.to_parquet(_BERT_TEST_PATH,   index=False)
    joblib.dump(svd_bert, _SVD_BERT_PATH, compress=3)
    print("✅ Embeddings y SVD guardados en disco")

# ── 7. Validación univariante — señal por componente ──────
print("\n" + "=" * 65)
print("  SEÑAL UNIVARIANTE — H Kruskal por componente BERT-SVD")
print("=" * 65)

componentes_bert = []
for col in cols_bert:
    grupos_k = [df_bert_train[col][y_train.values == k].values
                for k in sorted(y_train.unique())]
    H, _ = kruskal(*grupos_k)
    flag  = "✅" if H > 50 else ("⚠️ " if H > 10 else "🚨")
    print(f"  {flag} {col}  H={H:>8.1f}")
    if H > 10:
        componentes_bert.append(col)

print(f"\n  Componentes con señal (H>10): {len(componentes_bert)}/{N_COMPONENTES}")

# ── 8. Construir matrices de experimento ──────────────────
X_tr_bert = pd.concat([
    X_tr_exp5b.reset_index(drop=True),
    df_bert_train[componentes_bert].reset_index(drop=True),
], axis=1)

X_te_bert = pd.concat([
    X_te_exp5b.reset_index(drop=True),
    df_bert_test[componentes_bert].reset_index(drop=True),
], axis=1)

print(f"\n  X_tr_bert: {X_tr_bert.shape}  "
      f"(67 exp5b + {len(componentes_bert)} componentes BERT-SVD)")

# ── 9. CV — mismo bucle que EXP-NLP ──────────────────────
scores_bert = []
lgbm_bert   = LGBMClassifier(**params_modelo)

print("\n" + "=" * 65)
print(f"  CV — Exp5b + BERT ({X_tr_bert.shape[1]} features)")
print("=" * 65)

for fold, (idx_tr, idx_val) in enumerate(
        CV.split(X_tr_bert, y_train, groups=groups_train)):
    lgbm_bert.fit(
        X_tr_bert.iloc[idx_tr],  y_train.iloc[idx_tr] - 1,
        sample_weight      = sample_weights_train[idx_tr],
        eval_set           = [(X_tr_bert.iloc[idx_val], y_train.iloc[idx_val] - 1)],
        eval_sample_weight = [sample_weights_train[idx_val]],
        callbacks          = [lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(period=-1)],
    )
    score = macro_f1(y_train.iloc[idx_val],
                     lgbm_bert.predict(X_tr_bert.iloc[idx_val]) + 1)
    scores_bert.append(score)
    print(f"  Fold {fold+1} — Macro F1: {score:.4f}  "
          f"|  árboles: {lgbm_bert.best_iteration_}")

f1_bert   = np.mean(scores_bert)
delta_bert = f1_bert - f1_nlp   # comparación directa vs TF-IDF
flag_bert  = "✅" if delta_bert > 0.001 else ("⚠️ " if delta_bert >= 0 else "❌")

print(f"\n  {flag_bert} Exp5b + BERT ({X_tr_bert.shape[1]}f): "
      f"{f1_bert:.4f} ± {np.std(scores_bert):.4f}")
print(f"  Exp5b + TF-IDF         (82f): {f1_nlp:.4f}")
print(f"  Δ BERT vs TF-IDF           : {delta_bert:+.4f}")
print("=" * 65)

Cargando emilyalsentzer/Bio_ClinicalBERT...
  Dispositivo: cpu

Generando embeddings train (334,480 filas)...
        0/334480 procesados...
    80000/334480 procesados...
   160000/334480 procesados...
   240000/334480 procesados...
   320000/334480 procesados...

Generando embeddings test (83,620 filas)...
        0/83620 procesados...
    80000/83620 procesados...

Aplicando SVD (15 componentes)...
  Varianza explicada: 68.3%
✅ Embeddings y SVD guardados en disco

  SEÑAL UNIVARIANTE — H Kruskal por componente BERT-SVD
  ✅ bert_svd_00  H=  7916.2
  ✅ bert_svd_01  H= 11469.6
  ✅ bert_svd_02  H=  6277.8
  ✅ bert_svd_03  H= 15337.3
  ✅ bert_svd_04  H=  2647.6
  ✅ bert_svd_05  H= 17723.1
  ✅ bert_svd_06  H=  4056.2
  ✅ bert_svd_07  H=  5565.9
  ✅ bert_svd_08  H=  6649.0
  ✅ bert_svd_09  H=  2601.7
  ✅ bert_svd_10  H=  1169.2
  ✅ bert_svd_11  H=   648.2
  ✅ bert_svd_12  H=  3463.5
  ✅ bert_svd_13  H=  7749.2
  ✅ bert_svd_14  H=  3983.9

  Componentes con señal (H>10): 15/15

  X_tr_bert:

NameError: name 'f1_nlp' is not defined

In [31]:
# ── Guardar svd_bert en disco (ejecutar una vez si no existe) ──────────────
# Esta celda es un puente: guarda el svd_bert que ya está en memoria para
# que la celda anterior pueda cargarlo desde caché en futuras ejecuciones.
import joblib
_SVD_BERT_PATH = DATA_PROCESSED / "svd_bert.joblib"
if not _SVD_BERT_PATH.exists():
    joblib.dump(svd_bert, _SVD_BERT_PATH, compress=3)
    print(f"✅ svd_bert guardado en {_SVD_BERT_PATH}")
else:
    print(f"✅ svd_bert ya existe en {_SVD_BERT_PATH}")

✅ svd_bert ya existe en C:\Users\CARLOS\triaje-ia-tfg\data\processed\svd_bert.joblib


In [32]:
# ── Test final EXP-BERT ───────────────────────────────────
best_trees_bert = int(np.mean([lgbm_bert.best_iteration_]))

# Crear params sin n_estimators y sobreescribirlo
params_bert_final = {k: v for k, v in params_modelo.items() if k != "n_estimators"}

lgbm_bert_final = LGBMClassifier(**params_bert_final, n_estimators=best_trees_bert)
lgbm_bert_final.fit(
    X_tr_bert, y_train - 1,
    sample_weight=sample_weights_train,
)
y_pred_bert = lgbm_bert_final.predict(X_te_bert) + 1
f1_test_bert = macro_f1(y_test, y_pred_bert)

print(f"  Test F1 — Exp5b + BERT  : {f1_test_bert:.4f}")
print(f"  Test F1 — TF-IDF + TH   : 0.5615  (referencia)")
print(f"  Δ BERT vs TF-IDF+TH     : {f1_test_bert - 0.5615:+.4f}")

  Test F1 — Exp5b + BERT  : 0.5640
  Test F1 — TF-IDF + TH   : 0.5615  (referencia)
  Δ BERT vs TF-IDF+TH     : +0.0025


In [ ]:
# =========================================================
# EXP-BERT+NLP — TF-IDF + BERT-SVD combinados (97 features)
# =========================================================

# ── 1. Construir matrices ─────────────────────────────────
X_tr_bert_nlp = pd.concat([
    X_tr_nlp.reset_index(drop=True),       # 82f (exp5b + TF-IDF)
    df_bert_train.reset_index(drop=True),  # 15f BERT-SVD
], axis=1)

X_te_bert_nlp = pd.concat([
    X_te_nlp.reset_index(drop=True),
    df_bert_test.reset_index(drop=True),
], axis=1)

print(f"  X_tr_bert_nlp: {X_tr_bert_nlp.shape}  (esperado: 334480 × 97)")

# ── 2. CV ─────────────────────────────────────────────────
scores_bert_nlp = []
lgbm_bert_nlp   = LGBMClassifier(**params_modelo)

print("\n" + "=" * 65)
print(f"  CV — Exp5b + TF-IDF + BERT ({X_tr_bert_nlp.shape[1]} features)")
print("=" * 65)

for fold, (idx_tr, idx_val) in enumerate(
        CV.split(X_tr_bert_nlp, y_train, groups=groups_train)):
    lgbm_bert_nlp.fit(
        X_tr_bert_nlp.iloc[idx_tr],  y_train.iloc[idx_tr] - 1,
        sample_weight      = sample_weights_train[idx_tr],
        eval_set           = [(X_tr_bert_nlp.iloc[idx_val], y_train.iloc[idx_val] - 1)],
        eval_sample_weight = [sample_weights_train[idx_val]],
        callbacks          = [lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(period=-1)],
    )
    score = macro_f1(y_train.iloc[idx_val],
                     lgbm_bert_nlp.predict(X_tr_bert_nlp.iloc[idx_val]) + 1)
    scores_bert_nlp.append(score)
    print(f"  Fold {fold+1} — Macro F1: {score:.4f}  "
          f"|  árboles: {lgbm_bert_nlp.best_iteration_}")

f1_bert_nlp = np.mean(scores_bert_nlp)
print(f"\n  ✅ Exp5b + TF-IDF + BERT ({X_tr_bert_nlp.shape[1]}f): "
      f"{f1_bert_nlp:.4f} ± {np.std(scores_bert_nlp):.4f}")
print(f"  Exp5b + TF-IDF solo      (82f): {f1_nlp:.4f}")
print(f"  Exp5b + BERT solo        (82f): 0.5743")
print(f"  Δ combinado vs TF-IDF        : {f1_bert_nlp - f1_nlp:+.4f}")
print("=" * 65)

# ── 3. Test final ─────────────────────────────────────────
best_trees_bert_nlp  = int(np.mean([lgbm_bert_nlp.best_iteration_]))
params_bert_nlp_final = {k: v for k, v in params_modelo.items() if k != "n_estimators"}

lgbm_bert_nlp_final = LGBMClassifier(**params_bert_nlp_final,
                                      n_estimators=best_trees_bert_nlp)
lgbm_bert_nlp_final.fit(
    X_tr_bert_nlp, y_train - 1,
    sample_weight=sample_weights_train,
)
y_pred_bert_nlp  = lgbm_bert_nlp_final.predict(X_te_bert_nlp) + 1
f1_test_bert_nlp = macro_f1(y_test, y_pred_bert_nlp)

print(f"\n  Test F1 — TF-IDF + BERT  : {f1_test_bert_nlp:.4f}")
print(f"  Test F1 — TF-IDF + TH    : 0.5615  (referencia)")
print(f"  Δ combinado vs referencia: {f1_test_bert_nlp - 0.5615:+.4f}")

  X_tr_bert_nlp: (334480, 97)  (esperado: 334480 × 97)

  CV — Exp5b + TF-IDF + BERT (97 features)
  Fold 1 — Macro F1: 0.5769  |  árboles: 427
  Fold 2 — Macro F1: 0.5766  |  árboles: 445
  Fold 3 — Macro F1: 0.5802  |  árboles: 458
  Fold 4 — Macro F1: 0.5654  |  árboles: 417
  Fold 5 — Macro F1: 0.5861  |  árboles: 469

  ✅ Exp5b + TF-IDF + BERT (97f): 0.5770 ± 0.0067
  Exp5b + TF-IDF solo      (82f): 0.5722
  Exp5b + BERT solo        (82f): 0.5743
  Δ combinado vs TF-IDF        : +0.0048

  Test F1 — TF-IDF + BERT  : 0.5653
  Test F1 — TF-IDF + TH    : 0.5615  (referencia)
  Δ combinado vs referencia: +0.0038


In [ ]:
# ── Barrido de componentes BERT-SVD ──────────────────────
from sklearn.decomposition import TruncatedSVD

resultados_svd = []

for n_comp in [10, 15, 20, 30, 50]:
    # SVD con n componentes (fit solo en train)
    svd_k = TruncatedSVD(n_components=n_comp, random_state=42)
    tr_k  = svd_k.fit_transform(emb_train_full)
    te_k  = svd_k.transform(emb_test_full)

    cols_k   = [f"bert_{n_comp}c_{i:02d}" for i in range(n_comp)]
    df_tr_k  = pd.DataFrame(tr_k, columns=cols_k)
    df_te_k  = pd.DataFrame(te_k, columns=cols_k)

    X_tr_k = pd.concat([X_tr_exp5b.reset_index(drop=True), df_tr_k], axis=1)

    scores_k  = []
    lgbm_k    = LGBMClassifier(**params_modelo)

    for fold, (idx_tr, idx_val) in enumerate(
            CV.split(X_tr_k, y_train, groups=groups_train)):
        lgbm_k.fit(
            X_tr_k.iloc[idx_tr],  y_train.iloc[idx_tr] - 1,
            sample_weight      = sample_weights_train[idx_tr],
            eval_set           = [(X_tr_k.iloc[idx_val], y_train.iloc[idx_val] - 1)],
            eval_sample_weight = [sample_weights_train[idx_val]],
            callbacks          = [lgb.early_stopping(50, verbose=False),
                                   lgb.log_evaluation(period=-1)],
        )
        scores_k.append(macro_f1(y_train.iloc[idx_val],
                                  lgbm_k.predict(X_tr_k.iloc[idx_val]) + 1))

    f1_k     = np.mean(scores_k)
    var_k    = svd_k.explained_variance_ratio_.sum()
    resultados_svd.append({"n_comp": n_comp, "var_explicada": var_k, "cv_f1": f1_k})
    print(f"  n={n_comp:>2}  var={var_k:.1%}  CV F1={f1_k:.4f}")

# ── Tabla resumen ─────────────────────────────────────────
df_svd = pd.DataFrame(resultados_svd)
mejor  = df_svd.loc[df_svd["cv_f1"].idxmax()]
print(f"\n  Mejor configuración: n={int(mejor.n_comp)}  "
      f"F1={mejor.cv_f1:.4f}")
print(f"  Tu n=15 actual     :       F1=0.5743")

  n=10  var=59.8%  CV F1=0.5726
  n=15  var=68.3%  CV F1=0.5743
  n=20  var=73.8%  CV F1=0.5757
  n=30  var=80.3%  CV F1=0.5774
  n=50  var=86.9%  CV F1=0.5772

  Mejor configuración: n=30  F1=0.5774
  Tu n=15 actual     :       F1=0.5743


In [ ]:
# =========================================================
# GUARDAR MODELO BERT+NLP — bundle completo para inferencia
# =========================================================
# Incluye todo lo necesario para predecir sin re-ejecutar el notebook:
#   - lgbm_bert_nlp_final  → el clasificador
#   - tfidf                → TF-IDF vectorizer (ajustado en train)
#   - svd (TF-IDF SVD)     → reduce TF-IDF a 15 componentes
#   - svd_bert             → reduce embeddings BERT a 15 componentes
#   - componentes_bert     → lista de columnas BERT con señal (H>10)
#   - columnas del modelo  → orden exacto de features esperado
import joblib

bundle = {
    "modelo":              lgbm_bert_nlp_final,
    "tfidf":               tfidf,
    "svd_tfidf":           svd,
    "svd_bert":            svd_bert,
    "componentes_bert":    componentes_bert,
    "feature_cols_exp5b":  list(X_tr_exp5b.columns),
    "feature_cols_modelo": list(X_tr_bert_nlp.columns),
    "metrics": {
        "cv_f1_macro":   round(float(f1_bert_nlp), 6),
        "test_f1_macro": round(float(f1_test_bert_nlp), 6),
        "n_features":    int(X_tr_bert_nlp.shape[1]),
    },
}

ruta_bundle = MODELS_DIR / "lgbm_bert_nlp_bundle.joblib"
joblib.dump(bundle, ruta_bundle, compress=3)

print(f"Modelo guardado en : {ruta_bundle}")
print(f"  CV   Macro F1    : {f1_bert_nlp:.4f}")
print(f"  Test Macro F1    : {f1_test_bert_nlp:.4f}")
print(f"  Features         : {X_tr_bert_nlp.shape[1]}")
print(f"  Tamanyo archivo  : {ruta_bundle.stat().st_size / 1e6:.1f} MB")

NameError: name 'lgbm_bert_nlp_final' is not defined

In [ ]:
# =========================================================
# SHAP — Explicabilidad del modelo BERT+NLP (lgbm_bert_nlp_final)
# =========================================================
# Tiempo estimado con N_SHAP=2000 y LGBM: ~1-3 minutos
import shap
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg")  # evita problemas de display

# ── 1. Muestra aleatoria sobre train ──────────────────────
N_SHAP   = 2000
rng      = np.random.default_rng(42)
idx_shap = rng.choice(len(X_tr_bert_nlp), N_SHAP, replace=False)
X_shap   = X_tr_bert_nlp.iloc[idx_shap]

print(f"Calculando SHAP sobre {N_SHAP} muestras x {X_shap.shape[1]} features...")

# ── 2. TreeExplainer (nativo LGBM, muy rapido) ────────────
explainer   = shap.TreeExplainer(lgbm_bert_nlp_final)
shap_values = explainer.shap_values(X_shap, check_additivity=False)
# shape esperado: (N_SHAP, n_features, 5)

sv = np.array(shap_values)
print(f"  shap_values shape : {sv.shape}")

# ── 3. Guardar para no recalcular ─────────────────────────
shap_data = {
    "shap_values":    shap_values,
    "X_shap_np":      X_shap.values,
    "feature_names":  list(X_shap.columns),
    "expected_value": explainer.expected_value,
}
ruta_shap = NOTEBOOK_DIR / "resultados_shap_bert_nlp.pkl"
joblib.dump(shap_data, ruta_shap, compress=3)
print(f"  SHAP guardado en  : {ruta_shap}")

# ── 4. Importancia global — mean(|SHAP|) sobre muestras y clases ──
importancia = pd.Series(
    np.abs(sv).mean(axis=(0, 2)),
    index=list(X_shap.columns),
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 8))
importancia.head(20).plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("|SHAP| medio")
ax.set_title("Top 20 features — Importancia global SHAP (LGBM BERT+NLP)", fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
ruta_imp = "../../reports/figures/shap_importancia_bert_nlp.png"
plt.savefig(ruta_imp, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n  Top 5 features globales:")
for feat, val in importancia.head(5).items():
    print(f"    {feat:<35} {val:.5f}")

# ── 5. Beeswarm por clase ──────────────────────────────────
nombres_clase = {
    0: "Acuity 1 (critico)",
    1: "Acuity 2 (emergencia)",
    2: "Acuity 3 (urgente)",
    3: "Acuity 4 (menos urgente)",
    4: "Acuity 5 (no urgente)",
}

for clase_idx, nombre_clase in nombres_clase.items():
    expl_clase = shap.Explanation(
        values        = sv[:, :, clase_idx],
        base_values   = explainer.expected_value[clase_idx],
        data          = X_shap.values,
        feature_names = list(X_shap.columns),
    )
    shap.plots.beeswarm(expl_clase, max_display=15, show=False)
    plt.title(f"SHAP — {nombre_clase}", fontsize=12)
    plt.tight_layout()
    ruta_fig = f"../../reports/figures/shap_beeswarm_acuity{clase_idx + 1}_bert_nlp.png"
    plt.savefig(ruta_fig, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Figura guardada: {ruta_fig}")

print("\nSHAP completado.")

INCORPORACION DE LA IA (LLM)

In [33]:
# =========================================================
# CELDA EXPORT — Artefactos producción exp5b+BERT
# =========================================================
# Requiere haber ejecutado: CELDA 1, CELDA 2, CELDA 5, CELDA 36, CELDA 38.
#
# Exporta 5 artefactos a models/ para que TriajePredictor
# pueda cargarlos sin necesidad de re-ejecutar el notebook.
#
# Artefactos generados:
#   lgbm_exp5b_bert.joblib                 — clasificador final
#   exp5b_bert_features.json               — lista ordenada de features
#   exp5b_bert_thresholds.json             — pesos Nelder-Mead (5 clases)
#   bert_svd.joblib                        — TruncatedSVD para embeddings
#   exp5b_bert_production_assumptions.json — defaults e info para la UI
# =========================================================

import json
import joblib
import numpy as np
from scipy.optimize import minimize

# ── 1. Threshold tuning para el modelo BERT ────────────────────────────────
# OOF probas + Nelder-Mead (igual que EXP-TH pero sobre exp5b+BERT).
# Tiempo estimado: ~5-10 min (mismo orden que un CV normal).
# ───────────────────────────────────────────────────────────────────────────
print("Calculando OOF probas para threshold tuning del modelo BERT...")
print("(Puede tardar ~5-10 min — mismo tiempo que un fold de CV)\n")

oof_proba_bert = np.zeros((len(y_train), 5))
lgbm_th_bert   = LGBMClassifier(**params_modelo)

for fold, (idx_tr, idx_val) in enumerate(
        CV.split(X_tr_bert, y_train, groups=groups_train)):
    lgbm_th_bert.fit(
        X_tr_bert.iloc[idx_tr],  y_train.iloc[idx_tr] - 1,
        sample_weight      = sample_weights_train[idx_tr],
        eval_set           = [(X_tr_bert.iloc[idx_val], y_train.iloc[idx_val] - 1)],
        eval_sample_weight = [sample_weights_train[idx_val]],
        callbacks          = [lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(period=-1)],
    )
    oof_proba_bert[idx_val] = lgbm_th_bert.predict_proba(X_tr_bert.iloc[idx_val])
    s = macro_f1(y_train.iloc[idx_val], lgbm_th_bert.predict(X_tr_bert.iloc[idx_val]) + 1)
    print(f"  Fold {fold+1}/5  Macro F1: {s:.4f}  |  árboles: {lgbm_th_bert.best_iteration_}")

# Optimización Nelder-Mead
res_bert   = minimize(
    objetivo_th, x0=np.ones(5),
    args=(oof_proba_bert, y_train.values - 1),
    method="Nelder-Mead",
    options={"maxiter": 10000, "xatol": 1e-6, "fatol": 1e-6, "adaptive": True},
)
w_optimo        = res_bert.x
f1_sin_th       = macro_f1(y_train.values - 1, np.argmax(oof_proba_bert, axis=1))
f1_con_th       = -res_bert.fun

print(f"\n  OOF F1 sin thresholds : {f1_sin_th:.4f}")
print(f"  OOF F1 con thresholds : {f1_con_th:.4f}  (Δ={f1_con_th - f1_sin_th:+.4f})")
print(f"\n  Pesos óptimos : {[round(w, 4) for w in w_optimo]}")

# ── 2. Exportar 5 artefactos a models/ ────────────────────────────────────
print("\n" + "=" * 60)
print("  EXPORT → models/")
print("=" * 60)

# 2a. Clasificador final (entrenado sobre train completo en celda 38)
ruta_clf = MODELS_DIR / "lgbm_exp5b_bert.joblib"
joblib.dump(lgbm_bert_final, ruta_clf, compress=3)
print(f"  ✔ {ruta_clf.name:<42} ({ruta_clf.stat().st_size / 1e6:.1f} MB)")

# 2b. Lista ordenada de features (orden exacto de entrenamiento)
feature_dict = {"todas_features": list(X_tr_bert.columns)}
ruta_feat    = MODELS_DIR / "exp5b_bert_features.json"
with open(ruta_feat, "w", encoding="utf-8") as f:
    json.dump(feature_dict, f, indent=2)
print(f"  ✔ {ruta_feat.name:<42} ({len(feature_dict['todas_features'])} features)")

# 2c. Thresholds (pesos Nelder-Mead, 5 valores)
ruta_th = MODELS_DIR / "exp5b_bert_thresholds.json"
with open(ruta_th, "w", encoding="utf-8") as f:
    json.dump(w_optimo.tolist(), f)
print(f"  ✔ {ruta_th.name:<42} ({[round(w, 4) for w in w_optimo]})")

# 2d. SVD BERT (TruncatedSVD fit en train, sin leakage)
ruta_svd = MODELS_DIR / "bert_svd.joblib"
joblib.dump(svd_bert, ruta_svd, compress=3)
print(f"  ✔ {ruta_svd.name:<42} ({ruta_svd.stat().st_size / 1e3:.0f} KB)")

# 2e. Production assumptions (documentación de defaults de bloque 6)
production_assumptions = {
    "features_imputadas": [
        "n_visitas_previas", "visitas_ultimo_mes", "visitas_ultimo_año",
        "dias_desde_ultima_visita", "primera_visita", "frecuentador",
        "hx_cardiaco", "hx_respiratorio", "hx_neuro", "hx_psiquiatrico",
        "hx_abuso_sustancias", "hx_digestivo", "hx_metabolico_renal",
        "hx_infeccioso", "hx_trauma_muscular",
    ],
    "defaults": {
        "n_visitas_previas": 0,
        "primera_visita": 1,
        "dias_desde_ultima_visita": None,
    },
    "mensaje_ui": "Historial ED no disponible; bloque 6 imputado a valores conservadores.",
}
ruta_pa = MODELS_DIR / "exp5b_bert_production_assumptions.json"
with open(ruta_pa, "w", encoding="utf-8") as f:
    json.dump(production_assumptions, f, indent=2, ensure_ascii=False)
print(f"  ✔ {ruta_pa.name:<42} (ok)")

# ── 3. Validación de los artefactos exportados ────────────────────────────
print("\n" + "=" * 60)
print("  VALIDACIÓN")
print("=" * 60)

clf_v    = joblib.load(ruta_clf)
feats_v  = json.loads(ruta_feat.read_text())["todas_features"]
th_v     = json.loads(ruta_th.read_text())
svd_v    = joblib.load(ruta_svd)
pa_v     = json.loads(ruta_pa.read_text(encoding="utf-8"))

assert clf_v.n_features_in_ == len(feats_v), \
    f"Mismatch n_features_in_: {clf_v.n_features_in_} vs {len(feats_v)}"
assert len(th_v) == 5, f"Thresholds debe tener 5 pesos, tiene {len(th_v)}"
assert svd_v.n_components == svd_bert.n_components, "SVD n_components mismatch"
assert "mensaje_ui" in pa_v, "production_assumptions: falta mensaje_ui"

# Test de predicción sobre primera fila de test
X_fila = X_te_bert.iloc[:1].copy()
proba  = clf_v.predict_proba(X_fila)
assert proba.shape == (1, 5) and abs(proba[0].sum() - 1.0) < 1e-4

print(f"  ✔ n_features_in_  = {clf_v.n_features_in_}  (OK)")
print(f"  ✔ feature_list    = {len(feats_v)} columnas  (OK)")
print(f"  ✔ thresholds      = {[round(w, 4) for w in th_v]}  (OK)")
print(f"  ✔ SVD             = {svd_v.n_components} componentes  (OK)")
print(f"  ✔ predict_proba   → {[round(p, 3) for p in proba[0]]}  (OK)")
print(f"  ✔ assumptions     : {len(pa_v['features_imputadas'])} features imputadas  (OK)")
print(f"\n  Todo OK — artefactos listos para R3 (actualizar active_model.json)")


Calculando OOF probas para threshold tuning del modelo BERT...
(Puede tardar ~5-10 min — mismo tiempo que un fold de CV)

  Fold 1/5  Macro F1: 0.5730  |  árboles: 436
  Fold 2/5  Macro F1: 0.5779  |  árboles: 448
  Fold 3/5  Macro F1: 0.5773  |  árboles: 503
  Fold 4/5  Macro F1: 0.5669  |  árboles: 495
  Fold 5/5  Macro F1: 0.5798  |  árboles: 513

  OOF F1 sin thresholds : 0.5749
  OOF F1 con thresholds : 0.5787  (Δ=+0.0038)

  Pesos óptimos : [np.float64(0.7463), np.float64(1.0957), np.float64(1.1046), np.float64(0.8798), np.float64(1.1145)]

  EXPORT → models/
  ✔ lgbm_exp5b_bert.joblib                     (15.7 MB)
  ✔ exp5b_bert_features.json                   (82 features)
  ✔ exp5b_bert_thresholds.json                 ([np.float64(0.7463), np.float64(1.0957), np.float64(1.1046), np.float64(0.8798), np.float64(1.1145)])
  ✔ bert_svd.joblib                            (44 KB)
  ✔ exp5b_bert_production_assumptions.json     (ok)

  VALIDACIÓN
  ✔ n_features_in_  = 82  (OK)
  ✔ feat